# BDD4 — Empat Tugas Persepsi Jalan dalam Satu Model

**Google Colab · PyTorch · Bahasa Indonesia · eksperimen yang dapat direproduksi**

Notebook ini membangun **deteksi objek, semantic segmentation 19 kelas, lane marking segmentation biner, dan drivable area segmentation 3 kelas** dengan satu backbone bersama. Semua unduhan dataset berlangsung di mesin virtual Colab. Komputer lokal hanya perlu membuka notebook ini.

**Status:** implementasi telah diuji dengan fixture sintetis pada CPU, termasuk training singkat, resume, evaluasi, inferensi, dan ekspor. **Training/evaluasi penuh BDD100K dan pengujian GPU Colab belum dijalankan oleh penyusun.** Sel output eksperimen sengaja kosong. Tidak ada angka akurasi, kecepatan, atau jaminan generalisasi yang dibuat-buat.

**Cara menjalankan:** unggah notebook ini ke [Google Colab](https://colab.research.google.com/), pilih runtime GPU, lalu jalankan berurutan. Notebook mandiri: tiga modul implementasi ditulis oleh sel di bawah; file pendukung tidak perlu diunggah. Kaggle/Google dapat meminta autentikasi akun di dalam Colab. Gunakan folder Drive eksperimen baru bila mengubah konfigurasi, kode, atau dataset.

Training seluruh data, preprocessing poligon, evaluasi domain, dan penyimpanan ratusan ribu keluaran dapat melampaui satu sesi Colab. Checkpoint Drive mendukung kelanjutan lintas sesi. **Tidak ada mode default yang mengambil subset, `max_steps`, atau `drop_last=True`.** GPU/disk yang tidak cukup menyebabkan penghentian dengan pesan, bukan pengurangan dataset.

## 1. Environment Setup

Gunakan GPU T4/L4/A100 atau yang tersedia. Konfigurasi awal 384×640 dan batch 2 merupakan titik awal hemat memori; kemampuan menjalankannya diperiksa dengan forward/backward sungguhan di tahap 8. Detail garis dan objek kecil dapat membaik dengan resolusi lebih tinggi, tetapi perubahan itu memerlukan eksperimen tersendiri.

Versi PyTorch/torchvision dipasangkan secara eksplisit karena pipeline menggunakan API FCOS yang dapat berubah. Environment aktual, `pip freeze`, sumber kode, seed, fingerprint data, dan konfigurasi disimpan untuk reproduksi. CUDA dapat tetap memiliki operasi nondeterministik; seed tetap tidak sama dengan jaminan hasil bit-per-bit lintas GPU.

In [ ]:
import os, sys, subprocess, importlib.metadata
from pathlib import Path
assert 'google.colab' in sys.modules or Path('/content').exists(), 'Jalankan notebook ini di Google Colab.'
os.environ['MPLCONFIGDIR']='/content/bdd4/matplotlib_cache'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['OMP_NUM_THREADS']='2'
Path('/content/bdd4_lib').mkdir(parents=True,exist_ok=True)
Path('/content/bdd4/reports').mkdir(parents=True,exist_ok=True)

def installed(name):
    try: return importlib.metadata.version(name).split('+')[0]
    except importlib.metadata.PackageNotFoundError: return None

if installed('torch')!='2.9.0' or installed('torchvision')!='0.24.0':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','torch==2.9.0','torchvision==0.24.0',
                           '--index-url','https://download.pytorch.org/whl/cu126'])
packages=['kagglehub==0.3.13','numpy==2.2.6','pandas==2.2.3','matplotlib==3.10.3',
          'opencv-python-headless==4.11.0.86','ijson==3.4.0','tqdm==4.67.1',
          'pycocotools==2.0.10','requests==2.32.5','pillow==13.0.0']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*packages])
print('Dependensi terpasang. Bila runtime sebelumnya sudah mengimpor versi lain, restart sesi dan jalankan ulang.')


**Implementasi mandiri.** Ketiga sel berikut menulis source yang digunakan seluruh notebook. Modul data menangani audit, poligon Bezier, duplikasi dan label parsial; modul model mengimplementasikan empat head; modul engine menangani training, evaluasi, dan inferensi. Source yang sama tersedia terpisah di paket pendukung untuk ditinjau.

In [ ]:
%%writefile /content/bdd4_lib/bdd4_data.py
"""Audit, anotasi, split dan data loader BDD100K. Semua download berjalan di Colab."""
from __future__ import annotations
import concurrent.futures as cf
import hashlib
import json
import os
import random
import re
import shutil
import sqlite3
import warnings
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from urllib.parse import urljoin
import cv2
import ijson
import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

TASKS = ('det', 'sem', 'lane', 'drive')
DET_NAMES = ['pedestrian','rider','car','truck','bus','train','motorcycle','bicycle','traffic light','traffic sign']
SEM_NAMES = ['road','sidewalk','building','wall','fence','pole','traffic light','traffic sign','vegetation','terrain','sky','person','rider','car','truck','bus','train','motorcycle','bicycle']
DENSE_NAMES = {'sem': SEM_NAMES, 'lane':['background','lane marking'], 'drive':['direct','alternative','background']}
ALIASES = {'person':'pedestrian','bike':'bicycle','motor':'motorcycle'}
IGNORE = 255
SPLITS = ('train','val','test')
RANK = {'unknown':0,'train':1,'val':2,'test':3}
TOOLKIT_COMMIT = '9ac17c6c7c51d2fc83065fccd707cd5b1882a293'

@dataclass
class Config:
    seed: int = 2026
    work: str = '/content/bdd4'
    drive: str = '/content/drive/MyDrive/BDD4_research'
    kaggle_handle: str = 'solesensei/solesensei_bdd100k/versions/2'
    height: int = 384
    width: int = 640
    batch_size: int = 2
    workers: int = 2
    accumulation: int = 4
    epochs: int = 30
    lr: float = 2e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 2
    patience: int = 6
    min_delta: float = 1e-3
    checkpoint_steps: int = 500
    grad_clip: float = 5.0
    score_threshold: float = 0.25
    map_score_threshold: float = 0.001
    nms_threshold: float = 0.6
    max_detections: int = 100
    amp: bool = True
    download_official: bool = True
    strict_four_tasks: bool = True
    # URL portal resmi yang baru dapat ditempel di Colab jika server berpindah.
    official_urls: dict | None = None
    def dump(self):
        result=asdict(self);result['official_urls']=None  # URL portal bertoken tidak masuk laporan/checkpoint.
        return result

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Sebagian kernel deteksi CUDA tidak menjamin determinisme bit-per-bit.
    torch.use_deterministic_algorithms(True, warn_only=True)

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    def default(x):
        if isinstance(x, (np.integer,np.floating)): return x.item()
        if isinstance(x,np.ndarray): return x.tolist()
        if isinstance(x,Path): return str(x)
        raise TypeError(type(x).__name__)
    temporary.write_text(json.dumps(value, indent=2, default=default, allow_nan=False))
    temporary.replace(path)

def sha_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024**2), b''): h.update(block)
    return h.hexdigest()

def check_disk(path, required_gib):
    Path(path).mkdir(parents=True,exist_ok=True)
    free = shutil.disk_usage(path).free / 1024**3
    print(f'Disk {path}: tersedia {free:.1f} GiB, kebutuhan minimum tahap ini {required_gib:.1f} GiB')
    if free < required_gib:
        raise RuntimeError('Ruang disk tidak cukup. Ganti runtime/tujuan di Colab; jangan unduh ke komputer lokal.')

def split_of(path):
    # Exact component: trainA/testA bukan split. Ambil split terluar.
    return next((p for p in Path(path).parts if p in SPLITS), 'unknown')

def image_id(path):
    return re.sub(r'_(?:train|val|test)_(?:id|color)$','',Path(path).stem)

def classify_asset(path):
    s = '/' + str(path).replace('\\','/').lower()
    if Path(path).suffix.lower() in ('.jpg','.jpeg'):
        return 'image'
    if Path(path).suffix.lower() == '.png':
        if '/labels/sem_seg/masks/' in s or '/seg/labels/' in s: return 'sem'
        if '/labels/lane/masks/' in s: return 'lane'
        if '/labels/drivable/masks/' in s: return 'drive'
        if '/seg/images/' in s: return 'image'
        if '/images/' in s and not any(x in s for x in ('color_labels','colormaps')): return 'image'
    if Path(path).suffix.lower() == '.json' and (
        'bdd100k_labels_images_' in s or re.search(r'/det_(train|val|test)\.json$',s)
    ): return 'det_json'
    return 'other'

def inventory(roots, out):
    rows = []
    for root in roots:
        for path in sorted(Path(root).rglob('*')):
            if path.is_file():
                rows.append(dict(path=str(path),relative=str(path.relative_to(root)),root=str(root),
                                 kind=classify_asset(path),split=split_of(path.relative_to(root)),bytes=path.stat().st_size))
    frame = pd.DataFrame(rows)
    if frame.empty: raise RuntimeError('Dataset kosong')
    frame.to_csv(Path(out)/'asset_inventory.csv',index=False)
    print(frame.groupby(['kind','split']).agg(files=('path','size'),bytes=('bytes','sum')))
    return frame

OFFICIAL = {
 'lane': ('bdd100k_lane_labels_trainval.zip', 'dfe74f9ed6800765a0047414d620a186'),
 'drive': ('bdd100k_drivable_labels_trainval.zip','0abc320461200b1d7916f82fdcd64a96'),
 'sem': ('bdd100k_sem_seg_labels_trainval.zip','9a2968dde3345eeb689cffb1e26f9c78'),
}

def safe_unzip(archive, destination):
    destination = Path(destination).resolve(); destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        total = sum(i.file_size for i in z.infolist())
        check_disk(destination, total/1024**3 + 1)
        for item in z.infolist():
            target = (destination/item.filename).resolve()
            if not target.is_relative_to(destination) or ((item.external_attr>>16)&0o170000)==0o120000:
                raise ValueError('Path ZIP tidak aman')
        z.extractall(destination)  # zipfile memeriksa CRC ketika mengekstrak.

def retrieve_official(tasks, cfg):
    """Mencoba aset resmi; gagal secara terbuka dan menghasilkan log, tanpa mirror tidak resmi."""
    root = Path(cfg.work)/'official'; root.mkdir(parents=True,exist_ok=True)
    log = []
    # Sumber unduhan ini dinyatakan oleh dokumentasi resmi pada commit yang dicatat.
    base = 'https://dl.cv.ethz.ch/bdd100k/data/'
    index_links = []
    try:
        html = requests.get(base,timeout=(5,15)); html.raise_for_status()
        index_links = [urljoin(base,x) for x in re.findall(r'href=[\'\"]([^\'\"]+\.zip)[\'\"]',html.text)]
    except requests.RequestException as e:
        log.append({'asset':'index','status':'unavailable','error':type(e).__name__})
    for task in tasks:
        name, expected_md5 = OFFICIAL[task]
        candidates = list((cfg.official_urls or {}).get(task,[]))
        candidates += [u for u in index_links if name in u]
        candidates += [base+name]
        ok = False
        for url in dict.fromkeys(candidates):
            part = root/(task+'.zip.partial')
            try:
                # Jangan mencetak URL bertanda tangan/token.
                with requests.get(url,stream=True,timeout=(8,90)) as response:
                    response.raise_for_status()
                    required = int(response.headers.get('Content-Length',0))
                    check_disk(root, max(3, required/1024**3*3))
                    md5 = hashlib.md5()
                    with part.open('wb') as f:
                        for block in response.iter_content(4*1024**2):
                            f.write(block); md5.update(block)
                if md5.hexdigest() != expected_md5:
                    raise ValueError('Checksum berbeda dari rilis resmi; perbarui manifest resmi sebelum memakai aset ini')
                target = root/task
                safe_unzip(part,target)
                log.append({'asset':task,'status':'downloaded','md5':md5.hexdigest(),'sha256':sha_file(part)})
                ok = True; part.unlink(); break
            except Exception as e:
                part.unlink(missing_ok=True)
                log.append({'asset':task,'status':'failed_candidate','error':type(e).__name__})
        if not ok:
            log.append({'asset':task,'status':'missing_official_mask',
                        'action':'Gunakan poligon JSON yang valid, atau isi cfg.official_urls dari portal resmi dan jalankan ulang audit.'})
    write_json(Path(cfg.work)/'reports/official_retrieval.json',log)
    return root

def iter_records(path):
    with open(path,'rb') as f:
        first = f.read(128).lstrip(); f.seek(0)
        prefix = 'item' if first.startswith(b'[') else 'frames.item'
        yield from ijson.items(f,prefix,use_float=True)

def record_labels(record):
    if 'labels' in record: return record['labels']
    frames = record.get('frames',[])
    return frames[0].get('objects') if len(frames)==1 else None

def build_annotation_db(paths, database, report):
    database = Path(database)
    database.unlink(missing_ok=True)
    con = sqlite3.connect(database)
    con.execute('CREATE TABLE ann (id TEXT, split TEXT, payload TEXT, signature TEXT, source TEXT, PRIMARY KEY(id,split))')
    counts = Counter(); issues = []
    for path in tqdm(paths,desc='Streaming JSON (tanpa memuat seluruh file)'):
        split = split_of(path)
        if split=='unknown':
            split = next((s for s in SPLITS if re.search(r'[_/]'+s+r'(?:\.|_)',str(path))), 'unknown')
        if split=='unknown': raise ValueError(f'Split anotasi tidak diketahui: {path}')
        for r in iter_records(path):
            key = image_id(r['name']); labels = record_labels(r)
            if labels is None:
                issues.append({'id':key,'reason':'labels_missing_or_ambiguous_frames'}); continue
            boxes, poly = [], {'lane':[],'drive':[]}
            for label in labels:
                cat = ALIASES.get(label.get('category'),label.get('category'))
                if 'box2d' in label and cat not in DET_NAMES:
                    issues.append({'id':key,'reason':'unsupported_box_category','category':cat,'source':str(path)})
                if 'box2d' in label and cat in DET_NAMES:
                    b = label['box2d']; coords=[float(b[k]) for k in ('x1','y1','x2','y2')]
                    if not np.isfinite(coords).all() or coords[2]<=coords[0] or coords[3]<=coords[1]:
                        issues.append({'id':key,'reason':'invalid_box','source':str(path)}); continue
                    attrs = label.get('attributes') or {}
                    # Crowd/ignore disimpan, tidak dilatih sebagai objek positif.
                    ignore = bool(attrs.get('crowd') or attrs.get('ignore') or attrs.get('ignored'))
                    boxes.append({'box':coords,'label':DET_NAMES.index(cat),'ignore':ignore})
                    counts[cat]+=1
                p = label.get('poly2d')
                if p:
                    if label.get('category')=='lane': poly['lane'].append(label)
                    if label.get('category')=='drivable area': poly['drive'].append(label)
            payload={'boxes':boxes,'attributes':r.get('attributes') or {},'polygons':poly,
                     'video':r.get('videoName'),'source':str(path),'split':split}
            text=json.dumps(payload,sort_keys=True); signature=hashlib.sha256(text.encode()).hexdigest()
            old=con.execute('SELECT payload FROM ann WHERE id=? AND split=?',(key,split)).fetchone()
            if old:
                # Tidak memilih salah satu anotasi yang bertentangan secara diam-diam.
                prior=json.loads(old[0]); prior.pop('source',None); current=dict(payload);current.pop('source',None)
                if prior!=current: raise ValueError(f'Anotasi konflik: {key}/{split}')
            else: con.execute('INSERT INTO ann VALUES (?,?,?,?,?)',(key,split,text,signature,str(path)))
        con.commit()
    n=con.execute('SELECT count(*) FROM ann').fetchone()[0]; con.close()
    write_json(report,{'records':n,'box_counts':dict(counts),'issues':issues})
    # Box rusak dapat membuat gambar berlabel menjadi negative palsu; wajib diperbaiki.
    if any(x['reason'] in ('invalid_box','unsupported_box_category') for x in issues):
        raise ValueError('Bounding box invalid/kelas box tidak didukung ditemukan; lihat annotation_audit.json dan perbaiki sumber sebelum training.')
    return database

def audit_image(row):
    r=dict(row)
    try:
        with Image.open(r['path']) as im:
            im.load(); im=im.convert('RGB'); w,h=im.size
            raw=im.tobytes(); gray=np.asarray(im.convert('L').resize((32,32)),np.float32)
            spectrum=cv2.dct(gray)[:8,:8].ravel();bits=spectrum>np.median(spectrum[1:]);bits[0]=False
            phash=sum(int(b)<<j for j,b in enumerate(bits))
            thumbnail=np.asarray(im.resize((16,9)),np.uint8).tobytes().hex()
        r.update(valid=True,width=w,height=h,sha256=sha_file(r['path']),
                 pixels=hashlib.sha256(f'{w}x{h}:'.encode()+raw).hexdigest(),id=image_id(r['path']),phash=f'{phash:016x}',thumbnail=thumbnail)
    except Exception as e: r.update(valid=False,error=type(e).__name__,id=image_id(r['path']))
    return r

class UnionFind:
    def __init__(self,n): self.p=list(range(n))
    def find(self,a):
        while self.p[a]!=a: self.p[a]=self.p[self.p[a]]; a=self.p[a]
        return a
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a!=b:self.p[b]=a

def audit_images(assets, out, workers=4):
    rows=assets[assets.kind=='image'].to_dict('records')
    with cf.ThreadPoolExecutor(max_workers=workers) as pool:
        checked=list(tqdm(pool.map(audit_image,rows),total=len(rows),desc='Decode + SHA256 semua gambar'))
    images=pd.DataFrame(checked); images.to_csv(Path(out)/'physical_images.csv',index=False)
    valid=images[images.valid].copy().reset_index(drop=True)
    uf=UnionFind(len(valid))
    for col in ('id','sha256','pixels'):
        lookup={}
        for i,value in enumerate(valid[col]):
            if value in lookup:uf.union(i,lookup[value])
            else:lookup[value]=i
    # Pencarian konservatif near-duplicates: 5 band menjamin kandidat untuk Hamming <= 4.
    # Kandidat harus sangat dekat pula pada warna thumbnail; tidak menyamakan scene secara semantik.
    buckets=defaultdict(list);near=[];hashes=[int(x,16) for x in valid.phash]
    thumbs=[np.frombuffer(bytes.fromhex(x),np.uint8).astype(np.float32) for x in valid.thumbnail]
    bands=[(0,13),(13,13),(26,13),(39,13),(52,12)]
    for i,h in enumerate(tqdm(hashes,desc='Pemeriksaan near-duplicate')):
        candidates=set()
        for start,width in bands:candidates.update(buckets[(start,(h>>start)&((1<<width)-1))])
        for j in candidates:
            if uf.find(i)==uf.find(j):continue
            if (h^hashes[j]).bit_count()<=4 and valid.iloc[i]['width']==valid.iloc[j]['width'] and valid.iloc[i]['height']==valid.iloc[j]['height']:
                rmse=float(np.sqrt(np.mean((thumbs[i]-thumbs[j])**2)))
                if rmse<=1.5:
                    near.append({'first':valid.iloc[i].path,'second':valid.iloc[j].path,'phash_hamming':(h^hashes[j]).bit_count(),'thumbnail_rmse':rmse})
                    uf.union(i,j)
        for start,width in bands:buckets[(start,(h>>start)&((1<<width)-1))].append(i)
    write_json(Path(out)/'near_duplicate_candidates.json',near)
    valid['group']=[uf.find(i) for i in range(len(valid))]
    return images,valid

def decode_mask(path, task, encoding='official'):
    with Image.open(path) as im:
        a=np.asarray(im)
    if a.ndim!=2: raise ValueError(f'Mask harus single-channel ID, bukan colormap RGB: {path}')
    if task=='lane':
        if encoding=='binary':
            if not np.isin(a,[0,1,255]).all():raise ValueError('Mask lane binary invalid')
            return a.astype(np.uint8)
        # Kode evaluator resmi: foreground iff bit ke-3 == 0; 255 adalah background di aset lane.
        return (((a.astype(np.uint8)>>3)&1)==0).astype(np.uint8)
    n=19 if task=='sem' else 3
    if not np.isin(a,list(range(n))+[255]).all():
        raise ValueError(f'Encoding {task} bukan train IDs yang didukung: {path}; values={np.unique(a).tolist()}')
    return a.astype(np.uint8)

def audit_masks(assets, out):
    lookup=defaultdict(list); issues=[]
    for r in tqdm(assets[assets.kind.isin(['sem','lane','drive'])].to_dict('records'),desc='Validasi semua mask'):
        try:
            a=decode_mask(r['path'],r['kind']); h,w=a.shape
            if not (a!=255).any(): raise ValueError('Tidak ada piksel berlabel')
            key=(r['kind'],image_id(r['path']),r['split'])
            lookup[key].append({'path':r['path'],'width':w,'height':h,'encoding':'official',
                                'pixels':hashlib.sha256(a.tobytes()).hexdigest()})
        except Exception as e:issues.append({'path':r['path'],'error':str(e)})
    write_json(Path(out)/'mask_audit.json',{'valid':sum(map(len,lookup.values())),'invalid':issues})
    if issues: raise ValueError('Mask invalid ditemukan: lihat mask_audit.json; tidak boleh diabaikan diam-diam.')
    return lookup

def render_polygons(labels, task, shape, path):
    """Rasterisasi Bezier L/C: geometri asli, tanpa pseudo-label. Lebar garis 1 point, dpi=100."""
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    from matplotlib.path import Path as MPath
    from matplotlib.patches import PathPatch
    width,height=shape
    fig=Figure(figsize=(width/100,height/100),dpi=100,facecolor=(0,0,0))
    canvas=FigureCanvasAgg(fig); ax=fig.add_axes([0,0,1,1]); ax.set_axis_off()
    ax.set_xlim(0,width); ax.set_ylim(height,0)
    colors=[]
    for i,label in enumerate(labels):
        attrs=label.get('attributes') or {}
        if task=='drive':
            area=attrs.get('areaType')
            if area not in ('direct','alternative'): raise ValueError('areaType poligon drivable hilang')
            colors.append(0 if area=='direct' else 1)
        else: colors.append(1)
        for poly in label['poly2d']:
            vertices=list(poly['vertices']); types=poly['types']
            if len(types)!=len(vertices) or any(t not in 'LC' for t in types):raise ValueError('Format Bezier invalid')
            if not vertices: raise ValueError('Poligon kosong')
            codes=[MPath.LINETO if t=='L' else MPath.CURVE4 for t in types]; codes[0]=MPath.MOVETO
            # CURVE4 membutuhkan kelompok tiga control/end points.
            k=1
            while k<len(codes):
                if codes[k]==MPath.CURVE4:
                    if codes[k:k+3]!=[MPath.CURVE4]*3:raise ValueError('Bezier cubic tidak lengkap')
                    k+=3
                else:k+=1
            closed=task=='drive'
            if closed:vertices.append(vertices[0]);codes.append(MPath.LINETO)
            v=i+1; color=(((v>>8)&255)/255,(v&255)/255,0)
            ax.add_patch(PathPatch(MPath(vertices,codes),facecolor=color if closed else 'none',
                                   edgecolor=color,lw=0 if closed else 1,antialiased=False,snap=True))
    canvas.draw(); rgb=np.asarray(canvas.buffer_rgba())[...,:3].astype(np.uint16)
    indices=(rgb[...,0]<<8)+rgb[...,1]
    mask=np.full((height,width),2 if task=='drive' else 0,np.uint8)
    for i,color in enumerate(colors):mask[indices==i+1]=color
    Path(path).parent.mkdir(parents=True,exist_ok=True); Image.fromarray(mask).save(path)
    return mask

def prepare_manifest(valid, mask_lookup, db, cfg):
    con=sqlite3.connect(db); samples=[]; conflicts=[]; exclusions=[]
    generated=Path(cfg.work)/'generated_masks'
    for group, frame in tqdm(valid.groupby('group',sort=True),desc='Gabung label parsial + cegah leakage'):
        split=max(frame.split,key=lambda x:RANK[x])
        ids=sorted(set(frame.id)); primary=frame.sort_values('path').iloc[0]
        sample={'key':str(group),'path':primary.path,'id':primary.id,'ids':ids,'split':split,
                'width':int(primary.width),'height':int(primary.height),'image_sha':primary.pixels,'members':frame.path.tolist(),
                'det':None,'sem':None,'lane':None,'drive':None,'attributes':{},'video':None}
        # ID sama tetapi gambar berbeda: tidak memaksakan transfer geometri antar gambar.
        if frame.pixels.nunique()>1:
            conflicts.append({'group':int(group),'ids':ids,'reason':'same_ID_different_pixels'})
            sample['split']='quarantine'; samples.append(sample); continue
        for _,r in frame.iterrows():
            if r.split!=split:
                exclusions.append({'path':r.path,'official_split':r.split,'group_split':split,'reason':'cross_split_duplicate'})
        records=[]
        for key in ids:
            result=con.execute('SELECT payload FROM ann WHERE id=? AND split=?',(key,split)).fetchone()
            if result:records.append(json.loads(result[0]))
        if records:
            unique={json.dumps(r['boxes'],sort_keys=True) for r in records}
            if len(unique)>1:raise ValueError(f'Box konflik untuk duplikat {ids}')
            sample['det']=records[0]['boxes']; sample['attributes']=records[0]['attributes']; sample['video']=records[0]['video']
            for box in sample['det']:
                x1,y1,x2,y2=box['box']; box['box']=[max(0,x1),max(0,y1),min(sample['width'],x2),min(sample['height'],y2)]
                if box['box'][2]<=box['box'][0] or box['box'][3]<=box['box'][1]:raise ValueError('Box di luar gambar')
        for task in ('sem','lane','drive'):
            candidates=[m for key in ids for m in mask_lookup.get((task,key,split),[])]
            if candidates:
                if len({m['pixels'] for m in candidates})>1:raise ValueError(f'Mask konflik {task}/{ids}')
                chosen=candidates[0]
                if (chosen['width'],chosen['height'])!=(sample['width'],sample['height']):raise ValueError('Dimensi mask dan gambar berbeda')
                sample[task]=chosen
            elif task in ('lane','drive') and records:
                polygons=records[0]['polygons'][task]
                # Tanpa poligon dan tanpa mask: label tidak diketahui, BUKAN negative.
                if polygons:
                    geometry_hash=hashlib.sha256(json.dumps({'polygons':polygons,'size':[sample['width'],sample['height']]},sort_keys=True).encode()).hexdigest()[:16]
                    path=generated/task/split/(sample['id']+'_'+geometry_hash+'.png')
                    if path.exists():
                        a=decode_mask(path,task,'binary' if task=='lane' else 'official')
                        if a.shape!=(sample['height'],sample['width']):raise ValueError('Cache mask memiliki dimensi salah')
                    else:a=render_polygons(polygons,task,(sample['width'],sample['height']),path)
                    sample[task]={'path':str(path),'encoding':'binary' if task=='lane' else 'official',
                                  'source':'rasterized_Kaggle_polygons','pixels':hashlib.sha256(a.tobytes()).hexdigest()}
        samples.append(sample)
    con.close()
    # Kelompok video lintas split juga tidak boleh berkontribusi training/evaluasi.
    videos=defaultdict(list)
    for s in samples:
        if s['video']:videos[s['video']].append(s)
    for video,group in videos.items():
        if len({s['split'] for s in group})>1:
            for s in group:
                exclusions.append({'id':s['id'],'reason':'video_cross_split_conflict','video':video})
                s['split']='quarantine'
    report=Path(cfg.work)/'reports'
    write_json(report/'split_conflicts.json',{'conflicts':conflicts,'exclusions':exclusions})
    # Train/val/test disjoint pada ID, byte hash, decoded pixels, dan video yang tersedia.
    seen={}
    for s in samples:
        if s['split'] not in SPLITS:continue
        for key in s['ids']:
            if key in seen and seen[key]!=s['split']:raise AssertionError('Leakage ID')
            seen[key]=s['split']
    write_json(Path(cfg.work)/'manifest.json',samples)
    return samples

def coverage_report(images, valid, samples, cfg):
    rows=[]
    for split in (*SPLITS,'unknown','quarantine'):
        subset=[s for s in samples if s['split']==split]
        rows.append({'split':split,'unique_images':len(subset),
                     **{t:sum(s[t] is not None for s in subset) for t in TASKS}})
    table=pd.DataFrame(rows); table.to_csv(Path(cfg.work)/'reports/coverage.csv',index=False)
    missing=[{'id':s['id'],'split':s['split'],'task':t} for s in samples for t in TASKS if s[t] is None]
    pd.DataFrame(missing,columns=['id','split','task']).to_csv(Path(cfg.work)/'reports/missing_labels.csv',index=False)
    report={'physical_images':len(images),'valid_physical':len(valid),'invalid_physical':len(images)-len(valid),
            'canonical_images':len(samples),'duplicate_occurrences':len(valid)-len(samples),
            'coverage':rows,'policy':'all valid labels; no random truncation; missing tasks masked; cross-split groups assigned held-out precedence or quarantined',
            'near_duplicate_limit':'ID/byte/decoded-pixel/video checks plus pHash<=4 and thumbnail RMSE<=1.5 screen. Non-identical ambiguous groups quarantined. No guarantee for all near-duplicate views.'}
    write_json(Path(cfg.work)/'reports/dataset_coverage.json',report)
    print(table.to_string(index=False)); return table

def dataset_fingerprint(samples, assets, cfg):
    h=hashlib.sha256()
    for s in samples:
        # Lokasi cache dapat berubah antarsesi; fingerprint berdasarkan isi, ID, split, label.
        data={k:s[k] for k in ('key','id','ids','split','width','height','image_sha','det','attributes')}
        data['tasks']={t:None if s[t] is None else s[t].get('pixels') for t in ('sem','lane','drive')}
        h.update(json.dumps(data,sort_keys=True).encode())
    h.update(cfg.kaggle_handle.encode()); return h.hexdigest()

def training_statistics(samples,cfg):
    hist={t:np.zeros(len(DENSE_NAMES[t]),np.int64) for t in DENSE_NAMES}
    det=np.zeros(len(DET_NAMES),np.int64)
    for s in tqdm(samples,desc='Histogram hanya dari train'):
        if s['split']!='train':continue
        if s['det'] is not None:
            for b in s['det']:
                if not b.get('ignore'):det[b['label']]+=1
        for task in hist:
            if s[task] is not None:
                a=decode_mask(s[task]['path'],task,s[task]['encoding'])
                hist[task]+=np.bincount(a[a!=255],minlength=len(hist[task]))
    weights={}
    for task,h in hist.items():
        present=h>0; w=np.zeros(len(h),np.float32)
        if present.any():
            freq=h[present]/h[present].sum(); w[present]=1/np.sqrt(freq+1e-6)
            w[present]/=w[present].mean(); w[present]=np.clip(w[present],0.15,8)
        weights[task]=w.tolist()
    result={'dense_pixel_counts':{t:h.tolist() for t,h in hist.items()},'det_box_counts':det.tolist(),'class_weights':weights}
    write_json(Path(cfg.work)/'reports/train_class_statistics.json',result);return result

def letterbox(image, shape, mask=False):
    h,w=image.shape[:2]; H,W=shape; scale=min(W/w,H/h)
    nw,nh=round(w*scale),round(h*scale); left=(W-nw)//2; top=(H-nh)//2
    resized=cv2.resize(image,(nw,nh),interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_LINEAR)
    out=np.full((H,W) if image.ndim==2 else (H,W,3),255 if mask else 114,dtype=image.dtype)
    out[top:top+nh,left:left+nw]=resized
    return out,{'original':[h,w],'resized':[nh,nw],'pad':[top,left],'scale_xy':[nw/w,nh/h]}

class BDDDataset(Dataset):
    def __init__(self,samples,cfg,training=False):self.samples=samples;self.cfg=cfg;self.training=training;self.epoch=0
    def __len__(self):return len(self.samples)
    def __getitem__(self,index):
        s=self.samples[index]
        with Image.open(s['path']) as im:rgb=np.array(im.convert('RGB'))
        image,meta=letterbox(rgb,(self.cfg.height,self.cfg.width))
        boxes=[];labels=[];ignored=[]
        if s.get('det') is not None:
            for b in s['det']:
                xy=np.array(b['box'],np.float32).reshape(2,2)
                xy*=np.array(meta['scale_xy']);xy+=np.array(meta['pad'][::-1])
                if b.get('ignore'):ignored.append(xy.reshape(-1))
                else:boxes.append(xy.reshape(-1));labels.append(b['label'])
        masks={}
        for task in ('sem','lane','drive'):
            a=np.full((self.cfg.height,self.cfg.width),255,np.uint8)
            if s.get(task) is not None:
                original=decode_mask(s[task]['path'],task,s[task]['encoding'])
                if task=='lane':
                    # Occupancy downsampling mempertahankan garis tipis, hanya untuk training grid.
                    nh,nw=meta['resized'];top,left=meta['pad']
                    down=cv2.resize(original.astype(np.float32),(nw,nh),interpolation=cv2.INTER_AREA)
                    a[top:top+nh,left:left+nw]=(down>0).astype(np.uint8)
                else:a,_=letterbox(original,(self.cfg.height,self.cfg.width),mask=True)
            masks[task]=a
        boxes=np.asarray(boxes,np.float32).reshape(-1,4)
        ignored=np.asarray(ignored,np.float32).reshape(-1,4)
        if self.training:
            token=f'{self.cfg.seed}:{self.epoch}:{s["key"]}'.encode(); rng=np.random.default_rng(int.from_bytes(hashlib.sha256(token).digest()[:8],'little'))
            if rng.random()<0.5:
                image=image[:,::-1].copy();masks={t:a[:,::-1].copy() for t,a in masks.items()}
                if len(boxes):boxes[:,[0,2]]=self.cfg.width-boxes[:,[2,0]]
                if len(ignored):ignored[:,[0,2]]=self.cfg.width-ignored[:,[2,0]]
            gain=rng.uniform(0.75,1.25);offset=rng.uniform(-15,15)
            image=np.clip(image.astype(np.float32)*gain+offset,0,255).astype(np.uint8)
        target={'boxes':torch.as_tensor(boxes),'labels':torch.tensor(labels,dtype=torch.long),
                'det_valid':s.get('det') is not None,'ignore_boxes':torch.as_tensor(ignored),
                **{t:torch.from_numpy(a.astype(np.int64)) for t,a in masks.items()}}
        return torch.from_numpy(image.transpose(2,0,1).copy()).float()/255,target,{'sample':s,'transform':meta,'index':index}

def collate(batch):
    images,targets,meta=zip(*batch);return torch.stack(images),list(targets),list(meta)

def make_loader(dataset,cfg,epoch=0,start_batch=0,shuffle=False):
    dataset.epoch=epoch
    order=np.arange(len(dataset))
    if shuffle:np.random.default_rng(cfg.seed+epoch).shuffle(order)
    batches=[order[i:i+cfg.batch_size].tolist() for i in range(0,len(order),cfg.batch_size)]
    # Resumption melewati indeks, bukan membaca/melatih batch yang telah selesai.
    return DataLoader(dataset,batch_sampler=batches[start_batch:],num_workers=cfg.workers,
                      pin_memory=torch.cuda.is_available(),collate_fn=collate,generator=torch.Generator().manual_seed(cfg.seed+epoch),
                      persistent_workers=False,**({'prefetch_factor':2} if cfg.workers else {}))

def move_targets(targets,device):
    return [{k:v.to(device,non_blocking=True) if torch.is_tensor(v) else v for k,v in t.items()} for t in targets]


In [ ]:
%%writefile /content/bdd4_lib/bdd4_model.py
"""Satu ResNet50-FPN pretrained, FCOS, dan tiga decoder dense terpisah."""
from __future__ import annotations
import math
import torch
from torch import nn
import torch.nn.functional as F
from torchvision.models.detection import fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
from torchvision.models.detection.image_list import ImageList
from torchvision.models.detection.fcos import FCOSHead
from torchvision.ops import sigmoid_focal_loss, generalized_box_iou_loss
from torchvision.ops.misc import FrozenBatchNorm2d

class IgnoreAwareFCOSHead(FCOSHead):
    def compute_loss(self,targets,outputs,anchors,matched_idxs):
        logits=outputs['cls_logits']; classes=[]; gt=[]; valid=[]
        for t,a,m in zip(targets,anchors,matched_idxs):
            positive=m>=0
            labels=torch.full_like(m,-1); boxes=torch.zeros_like(a)
            if positive.any():
                labels[positive]=t['labels'][m[positive]];boxes[positive]=t['boxes'][m[positive]]
            allowed=torch.ones_like(positive)
            ignored=t.get('ignore_boxes',a.new_empty((0,4)))
            if len(ignored):
                center=(a[:,:2]+a[:,2:])/2
                inside=((center[:,None]>=ignored[None,:,:2]) & (center[:,None]<=ignored[None,:,2:])).all(-1).any(-1)
                allowed=~inside | positive
            classes.append(labels);gt.append(boxes);valid.append(allowed)
        classes=torch.stack(classes);gt=torch.stack(gt);a=torch.stack(anchors)
        foreground=classes>=0;normalizer=foreground.sum().clamp_min(1)
        one_hot=torch.zeros_like(logits)
        positions=foreground.nonzero(as_tuple=True)
        one_hot[positions[0],positions[1],classes[foreground]]=1
        focal=sigmoid_focal_loss(logits,one_hot,reduction='none').sum(-1)
        classification=(focal*torch.stack(valid)).sum()/normalizer
        decoded=self.box_coder.decode(outputs['bbox_regression'],a)
        regression=generalized_box_iou_loss(decoded[foreground],gt[foreground],reduction='sum')/normalizer
        encoded=self.box_coder.encode(a,gt)[foreground]
        if encoded.numel():
            lr=encoded[:,[0,2]];tb=encoded[:,[1,3]]
            center=((lr.min(-1).values/lr.max(-1).values.clamp_min(1e-7))*(tb.min(-1).values/tb.max(-1).values.clamp_min(1e-7))).clamp_min(0).sqrt()
        else:center=encoded.new_empty((0,))
        centerness=F.binary_cross_entropy_with_logits(outputs['bbox_ctrness'].squeeze(-1)[foreground],center,reduction='sum')/normalizer
        return {'classification':classification,'bbox_regression':regression,'bbox_ctrness':centerness}

def frozen_batch_norm(module):
    for name,child in list(module.named_children()):
        if isinstance(child,nn.BatchNorm2d):
            replacement=FrozenBatchNorm2d(child.num_features,eps=child.eps)
            with torch.no_grad():
                for field in ('weight','bias','running_mean','running_var'):getattr(replacement,field).copy_(getattr(child,field))
            setattr(module,name,replacement)
        else:frozen_batch_norm(child)

from bdd4_data import TASKS, DET_NAMES

class DenseDecoder(nn.Module):
    def __init__(self,classes,width=64):
        super().__init__()
        self.projections=nn.ModuleList([nn.Sequential(nn.Conv2d(256,width,1,bias=False),nn.GroupNorm(8,width),nn.ReLU()) for _ in range(4)])
        self.fuse=nn.Sequential(nn.Conv2d(width*4,width,3,padding=1,bias=False),nn.GroupNorm(8,width),nn.ReLU(),
                                nn.Dropout2d(0.1),nn.Conv2d(width,width,3,padding=1,bias=False),nn.GroupNorm(8,width),nn.ReLU(),nn.Conv2d(width,classes,1))
    def forward(self,features,shape):
        size=features[0].shape[-2:]
        x=[F.interpolate(p(f),size=size,mode='bilinear',align_corners=False) for p,f in zip(self.projections,features)]
        return F.interpolate(self.fuse(torch.cat(x,1)),size=shape,mode='bilinear',align_corners=False)

class FourTaskNet(nn.Module):
    def __init__(self,pretrained=True,max_detections=100):
        super().__init__()
        weights=FCOS_ResNet50_FPN_Weights.COCO_V1 if pretrained else None
        self.detector=fcos_resnet50_fpn(weights=weights,weights_backbone=None,trainable_backbone_layers=5,
                                        score_thresh=0.001,nms_thresh=0.6,detections_per_img=max_detections)
        if weights is None:frozen_batch_norm(self.detector)
        replacement=IgnoreAwareFCOSHead(256,1,91)
        replacement.load_state_dict(self.detector.head.state_dict())
        self.detector.head=replacement
        # Ekstrak C2 tambahan tanpa mengeksekusi backbone dua kali.
        self.detector.backbone.body.return_layers['layer1']='c2'
        old=self.detector.head.classification_head.cls_logits
        head=self.detector.head.classification_head
        head.num_classes=len(DET_NAMES)
        head.cls_logits=nn.Conv2d(old.in_channels,len(DET_NAMES),3,padding=1)
        nn.init.normal_(head.cls_logits.weight,std=0.01)
        nn.init.constant_(head.cls_logits.bias,-math.log((1-0.01)/0.01))
        if weights is not None:
            categories=weights.meta['categories']
            # Rider mendapatkan initialisasi person; kelasnya tetap dilatih terpisah.
            mapping={'pedestrian':'person','rider':'person','traffic sign':'stop sign'}
            with torch.no_grad():
                for i,name in enumerate(DET_NAMES):
                    source=categories.index(mapping.get(name,name))
                    head.cls_logits.weight[i].copy_(old.weight[source]);head.cls_logits.bias[i].copy_(old.bias[source])
        self.c2_projection=nn.Conv2d(256,256,1)
        self.dense_heads=nn.ModuleDict({'sem':DenseDecoder(19),'lane':DenseDecoder(2),'drive':DenseDecoder(3)})
        self.log_vars=nn.Parameter(torch.zeros(4))
        self.register_buffer('mean',torch.tensor([0.485,0.456,0.406])[None,:,None,None])
        self.register_buffer('std',torch.tensor([0.229,0.224,0.225])[None,:,None,None])
    def raw(self,images):
        normalized=(images-self.mean)/self.std
        body=self.detector.backbone.body(normalized)
        c2=body.pop('c2')
        features=list(self.detector.backbone.fpn(body).values())
        p2=self.c2_projection(c2)+F.interpolate(features[0],size=c2.shape[-2:],mode='nearest')
        dense_features=[p2,*features[:3]]
        dense={name:head(dense_features,images.shape[-2:]) for name,head in self.dense_heads.items()}
        raw=self.detector.head(features)
        shapes=[tuple(images.shape[-2:]) for _ in range(images.shape[0])]
        anchors=self.detector.anchor_generator(ImageList(images,shapes),features)
        counts=[f.shape[-2]*f.shape[-1] for f in features]
        return raw,dense,anchors,counts
    def forward(self,images,targets=None,predict=False):
        raw,dense,anchors,counts=self.raw(images)
        losses={}
        if targets is not None:
            indices=[i for i,t in enumerate(targets) if t['det_valid']]
            if indices:
                # Loss deteksi HANYA menerima gambar berlabel deteksi; unlabeled bukan negative.
                subset={k:v[indices].float() for k,v in raw.items()}
                parts=self.detector.compute_loss([targets[i] for i in indices],subset,[anchors[i] for i in indices],counts)
                losses['det']=sum(parts.values())
        detections=None
        if predict:
            split={k:list(v.float().split(counts,dim=1)) for k,v in raw.items()}
            detections=self.detector.postprocess_detections(split,[list(a.split(counts)) for a in anchors],
                                                           [tuple(images.shape[-2:])]*images.shape[0])
        return {'dense':dense,'losses':losses,'detections':detections}

class MultiTaskCriterion(nn.Module):
    def __init__(self,weights):
        super().__init__()
        for task,values in weights.items():self.register_buffer('w_'+task,torch.tensor(values,dtype=torch.float32))
    def forward(self,model,output,targets):
        losses=dict(output['losses']); counts={'det':sum(t['det_valid'] for t in targets)}
        for task,logits in output['dense'].items():
            target=torch.stack([t[task] for t in targets]);valid=target!=255
            active=valid.flatten(1).any(1); counts[task]=int(active.sum())
            if not active.any():continue
            logits=logits[active].float();target=target[active];valid=target!=255
            weight=getattr(self,'w_'+task)
            # Satu mean per gambar lalu mean per tugas; gambar tanpa label tidak masuk penyebut.
            ce=F.cross_entropy(logits,target,weight=weight,ignore_index=255,reduction='none')
            ce=(ce.sum((1,2))/valid.sum((1,2)).clamp_min(1)).mean()
            one=F.one_hot(target.masked_fill(~valid,0),logits.shape[1]).permute(0,3,1,2).float()*valid[:,None]
            prob=logits.softmax(1)*valid[:,None]
            intersection=(prob*one).sum((0,2,3));denominator=(prob+one).sum((0,2,3))
            present=one.sum((0,2,3))>0
            if task=='lane':present[0]=False  # Dice foreground saja.
            dice=(1-(2*intersection[present]+1)/(denominator[present]+1)).mean() if present.any() else logits.sum()*0
            losses[task]=ce+0.5*dice
        if not losses:raise RuntimeError('Batch tidak memiliki label tugas apa pun')
        total=next(iter(losses.values()))*0
        for i,task in enumerate(TASKS):
            if task in losses:
                # Homoscedastic uncertainty: regularizer juga hanya aktif untuk tugas berlabel.
                s=model.log_vars[i]
                total=total+torch.exp(-s)*losses[task]+s
        return total,losses,counts

class ExportRaw(nn.Module):
    """Kontrak ekspor: 3 tensor FCOS, anchor, 3 logits dense. NMS di luar graph."""
    def __init__(self,model):super().__init__();self.model=model
    def forward(self,images):
        raw,dense,anchors,counts=self.model.raw(images)
        return (raw['cls_logits'],raw['bbox_regression'],raw['bbox_ctrness'],torch.stack(anchors),
                dense['sem'],dense['lane'],dense['drive'])


In [ ]:
%%writefile /content/bdd4_lib/bdd4_engine.py
"""Training resumable, evaluasi asli, inferensi seluruh gambar, dan ekspor."""
from __future__ import annotations
import contextlib
import csv
import hashlib
import io
import json
import math
import os
import random
import shutil
import time
from collections import defaultdict
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision.ops import box_iou
from tqdm.auto import tqdm
from bdd4_data import (TASKS,DET_NAMES,SEM_NAMES,DENSE_NAMES,BDDDataset,make_loader,move_targets,
                       write_json,decode_mask,check_disk,sha_file)
from bdd4_model import FourTaskNet,MultiTaskCriterion,ExportRaw

def autocast_context(cfg,device):
    enabled=cfg.amp and device.type=='cuda'
    dtype=torch.bfloat16 if enabled and torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type=device.type,dtype=dtype,enabled=enabled)

def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(state):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda'] is not None:torch.cuda.set_rng_state_all(state['cuda'])

def atomic_checkpoint(path,state):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    torch.save(state,tmp);tmp.replace(path)

def experiment_signature(cfg,fingerprint):
    contract={k:v for k,v in cfg.dump().items() if k not in ('work','drive','official_urls','workers','download_official')}
    sources={name:sha_file(Path(__file__).with_name(name)) for name in ('bdd4_data.py','bdd4_model.py','bdd4_engine.py')}
    return hashlib.sha256(json.dumps({'config':contract,'data':fingerprint,'source':sources,'torch':torch.__version__},sort_keys=True).encode()).hexdigest()

def aggregate_losses(sums,counts,losses,active):
    for task,value in losses.items():
        n=active[task];sums[task]+=float(value.detach())*n;counts[task]+=n

def averages(sums,counts):return {t:sums[t]/counts[t] if counts[t] else None for t in TASKS}

@torch.no_grad()
def validation_losses(model,criterion,dataset,cfg,device):
    model.eval();sums=defaultdict(float);counts=defaultdict(int)
    for images,targets,metadata in tqdm(make_loader(dataset,cfg),desc='Validasi seluruh label',leave=False):
        images=images.to(device,non_blocking=True);targets=move_targets(targets,device)
        with autocast_context(cfg,device):
            output=model(images,targets);total,losses,active=criterion(model,output,targets)
        aggregate_losses(sums,counts,losses,active)
    return averages(sums,counts)

def train(model,samples,statistics,cfg,fingerprint,device,resume=True):
    training=[s for s in samples if s['split']=='train' and any(s[t] is not None for t in TASKS)]
    validation=[s for s in samples if s['split']=='val' and any(s[t] is not None for t in TASKS)]
    if not training or not validation:raise RuntimeError('Train/val berlabel harus tersedia')
    for task in TASKS:
        if cfg.strict_four_tasks and not any(s[task] is not None for s in training):
            raise RuntimeError(f'Tugas {task} tidak memiliki label train. Ambil aset resmi; training empat tugas dihentikan.')
        if cfg.strict_four_tasks and not any(s[task] is not None for s in validation):
            raise RuntimeError(f'Tugas {task} tidak memiliki label val untuk model selection.')
    train_ds=BDDDataset(training,cfg,True);val_ds=BDDDataset(validation,cfg,False)
    model=model.to(device);criterion=MultiTaskCriterion(statistics['class_weights']).to(device)
    backbone_ids={id(p) for p in model.detector.backbone.parameters()}
    groups=[{'params':[p for p in model.parameters() if id(p) in backbone_ids], 'lr':cfg.lr*0.1},
            {'params':[p for p in model.parameters() if id(p) not in backbone_ids and p is not model.log_vars], 'lr':cfg.lr},
            {'params':[model.log_vars], 'lr':cfg.lr,'weight_decay':0.0}]
    optimizer=torch.optim.AdamW(groups,weight_decay=cfg.weight_decay)
    def schedule(epoch):
        if epoch<cfg.warmup_epochs:return (epoch+1)/max(1,cfg.warmup_epochs)
        phase=(epoch-cfg.warmup_epochs)/max(1,cfg.epochs-cfg.warmup_epochs)
        return 0.05+0.95*0.5*(1+math.cos(math.pi*phase))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,schedule)
    scaler=torch.amp.GradScaler('cuda',enabled=cfg.amp and device.type=='cuda' and not torch.cuda.is_bf16_supported())
    signature=experiment_signature(cfg,fingerprint)
    checkpoint_dir=Path(cfg.drive)/'checkpoints';checkpoint_dir.mkdir(parents=True,exist_ok=True)
    last_path=checkpoint_dir/'last.pt';best_path=checkpoint_dir/'best.pt'
    state={'epoch':0,'next_batch':0,'history':[],'best':float('inf'),'bad_epochs':0,'scales':None,
           'sums':{},'counts':{},'seen':[],'global_step':0,'finished':False}
    if resume and last_path.exists():
        saved=torch.load(last_path,map_location='cpu',weights_only=False)  # checkpoint milik eksperimen sendiri.
        if saved['signature']!=signature:raise RuntimeError('Config/dataset berbeda. Gunakan folder eksperimen baru untuk mencegah resume yang salah.')
        model.load_state_dict(saved['model']);optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler']);scaler.load_state_dict(saved['scaler']);state=saved['progress'];restore_rng(saved['rng'])
        # Optimizer state dipindahkan ke GPU setelah load.
        for value in optimizer.state.values():
            for k,v in value.items():
                if torch.is_tensor(v):value[k]=v.to(device)
        print(f'Resume epoch {state["epoch"]+1}, batch berikutnya {state["next_batch"]}, step {state["global_step"]}')
    def save(path):
        atomic_checkpoint(path,{'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                                'scaler':scaler.state_dict(),'progress':state,'rng':rng_state(),'config':cfg.dump(),
                                'fingerprint':fingerprint,'signature':signature,'torch':torch.__version__})
    if state['finished']:
        print('Training sebelumnya telah selesai/early stopped. Memuat best checkpoint.');return load_best(model,cfg,device),state['history']
    total_batches=math.ceil(len(train_ds)/cfg.batch_size)
    try:
        for epoch in range(state['epoch'],cfg.epochs):
            model.train();optimizer.zero_grad(set_to_none=True)
            start=state['next_batch'] if epoch==state['epoch'] else 0
            sums=defaultdict(float,state['sums']);counts=defaultdict(int,state['counts']);seen=set(state['seen'])
            loader=make_loader(train_ds,cfg,epoch,start,shuffle=True)
            started=time.monotonic();window=0;window_inputs=[];window_rng=None
            for batch_id,(images,targets,metadata) in enumerate(tqdm(loader,total=total_batches-start,desc=f'Epoch {epoch+1}/{cfg.epochs}'),start):
                if window==0:window_rng=rng_state();window_inputs=[]
                window_inputs.append((images,targets))  # CPU cache untuk replay jika FP16 gradient overflow.
                images=images.to(device,non_blocking=True);targets=move_targets(targets,device)
                # Ukuran window akumulasi terakhir dapat lebih pendek.
                divisor=min(cfg.accumulation,total_batches-(batch_id//cfg.accumulation)*cfg.accumulation)
                with autocast_context(cfg,device):
                    output=model(images,targets);total,losses,active=criterion(model,output,targets)
                    loss=total/divisor
                if not torch.isfinite(loss):raise FloatingPointError(f'Loss non-finite pada epoch {epoch+1}, batch {batch_id}')
                scaler.scale(loss).backward();window+=1
                aggregate_losses(sums,counts,losses,active)
                seen.update(m['index'] for m in metadata)
                boundary=(batch_id+1)%cfg.accumulation==0 or batch_id+1==total_batches
                if boundary:
                    for retry in range(7):
                        scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip)
                        if torch.isfinite(norm):
                            scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True);break
                        if not scaler.is_enabled() or retry==6:raise FloatingPointError('Gradien non-finite setelah AMP replay')
                        # GradScaler melewati optimizer step yang overflow, lalu window direplay.
                        scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
                        state['amp_replays']=state.get('amp_replays',0)+1
                        print('FP16 gradient overflow: menurunkan scale dan mengulang window; tidak membuang gambar.')
                        restore_rng(window_rng)
                        for cpu_images,cpu_targets in window_inputs:
                            with autocast_context(cfg,device):
                                replay_output=model(cpu_images.to(device),move_targets(cpu_targets,device))
                                replay_total,_,_=criterion(model,replay_output,move_targets(cpu_targets,device))
                            scaler.scale(replay_total/divisor).backward()
                    with torch.no_grad():model.log_vars.clamp_(-3,3)
                    state.update(epoch=epoch,next_batch=batch_id+1,sums=dict(sums),counts=dict(counts),seen=sorted(seen),global_step=state['global_step']+1)
                    if state['global_step']%cfg.checkpoint_steps==0:save(last_path)
                    window=0;window_inputs=[];window_rng=None
            if len(seen)!=len(train_ds):raise AssertionError('Coverage epoch tidak lengkap; ada gambar train yang belum dipakai')
            save(last_path)  # Cursor akhir train tersimpan sebelum validasi yang dapat memakan waktu lama.
            train_loss=averages(sums,counts);val_loss=validation_losses(model,criterion,val_ds,cfg,device)
            if state['scales'] is None:state['scales']={t:max(v,1e-8) for t,v in val_loss.items() if v is not None}
            score=float(np.mean([math.log1p(val_loss[t]/state['scales'][t]) for t in state['scales']]))
            row={'epoch':epoch+1,'train':train_loss,'val':val_loss,'selection_score':score,
                 'seconds':time.monotonic()-started,'unique_train_images_seen':len(seen),
                 'task_train_images':dict(counts),'uncertainty_weights':torch.exp(-model.log_vars.detach()).cpu().tolist(),
                 'lr':[g['lr'] for g in optimizer.param_groups]}
            state['history'].append(row)
            improved=score<state['best']-cfg.min_delta
            if improved:state['best']=score;state['bad_epochs']=0
            else:state['bad_epochs']+=1
            scheduler.step();state.update(epoch=epoch+1,next_batch=0,sums={},counts={},seen=[])
            write_json(Path(cfg.drive)/'reports/history.json',state['history'])
            if improved:save(best_path)
            state['finished']=state['bad_epochs']>=cfg.patience or epoch+1>=cfg.epochs
            save(last_path)
            print(json.dumps(row,indent=2))
            if state['finished']:
                print('Early stopping' if state['bad_epochs']>=cfg.patience else 'Mencapai jumlah epoch terkonfigurasi');break
    except torch.cuda.OutOfMemoryError:
        # Tidak menurunkan resolusi atau membuang batch diam-diam.
        print('GPU OOM. Checkpoint optimizer-boundary terakhir tetap aman. Pilih GPU lebih besar atau eksperimen baru dengan batch lebih kecil.')
        raise
    return load_best(model,cfg,device),state['history']

def load_best(model,cfg,device):
    path=Path(cfg.drive)/'checkpoints/best.pt'
    if not path.exists():raise FileNotFoundError('Best checkpoint belum tersedia; jalankan training terlebih dahulu.')
    saved=torch.load(path,map_location='cpu',weights_only=False);model.load_state_dict(saved['model']);return model.to(device).eval()

def unletterbox_logits(logits,meta):
    top,left=meta['pad'];nh,nw=meta['resized']
    crop=logits[:,top:top+nh,left:left+nw]
    return F.interpolate(crop[None].float(),size=meta['original'],mode='bilinear',align_corners=False)[0]

def restore_boxes(pred,meta):
    boxes=pred['boxes'].detach().float().cpu().clone();top,left=meta['pad'];sx,sy=meta['scale_xy'];h,w=meta['original']
    boxes[:,[0,2]]=(boxes[:,[0,2]]-left)/sx;boxes[:,[1,3]]=(boxes[:,[1,3]]-top)/sy
    boxes[:,[0,2]].clamp_(0,w);boxes[:,[1,3]].clamp_(0,h)
    keep=(boxes[:,2]>boxes[:,0]) & (boxes[:,3]>boxes[:,1])
    return {'boxes':boxes[keep],'scores':pred['scores'].detach().cpu()[keep],'labels':pred['labels'].detach().cpu()[keep]}

def confusion_update(matrix,target,pred):
    valid=target!=255;n=matrix.shape[0]
    matrix+=np.bincount((target[valid].astype(np.int64)*n+pred[valid]).ravel(),minlength=n*n).reshape(n,n)

def dense_scores(matrix,foreground=None):
    matrix=matrix.astype(np.float64);tp=np.diag(matrix);gt=matrix.sum(1);pd=matrix.sum(0)
    union=gt+pd-tp;denom=gt+pd
    iou=np.divide(tp,union,out=np.full(len(tp),np.nan),where=union>0)
    dice=np.divide(2*tp,denom,out=np.full(len(tp),np.nan),where=denom>0)
    idx=np.arange(len(tp)) if foreground is None else np.array(foreground)
    mean=lambda a:float(np.nanmean(a[idx])) if np.isfinite(a[idx]).any() else None
    clean=lambda a:[float(v) if np.isfinite(v) else None for v in a]
    result={'mean_iou':mean(iou),'dice':mean(dice),'per_class_iou':clean(iou),'per_class_dice':clean(dice),'confusion':matrix.astype(np.int64).tolist()}
    if len(tp)==2:
        precision=tp[1]/pd[1] if pd[1] else 0.0;recall=tp[1]/gt[1] if gt[1] else 0.0
        result.update(precision=float(precision),recall=float(recall),f1=float(2*precision*recall/(precision+recall)) if precision+recall else 0.0,iou=float(iou[1]) if np.isfinite(iou[1]) else None)
    return result

def lane_tolerance_counts(target,pred,radius):
    kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*radius+1,2*radius+1))
    gt=(target==1).astype(np.uint8);pd=(pred==1).astype(np.uint8)
    return np.array([(pd*cv2.dilate(gt,kernel)).sum(),pd.sum(),(gt*cv2.dilate(pd,kernel)).sum(),gt.sum()],np.int64)

def detection_counts(pred,gt,threshold=0.25,iou_threshold=0.5):
    """Micro precision/recall: greedy matching per kelas, threshold ditetapkan sebelum testing."""
    select=pred['scores']>=threshold;boxes=pred['boxes'][select];labels=pred['labels'][select]
    order=pred['scores'][select].argsort(descending=True);boxes=boxes[order];labels=labels[order]
    regular=[b for b in gt if not b.get('ignore')];ignored=[b for b in gt if b.get('ignore')]
    gt_boxes=torch.tensor([b['box'] for b in regular],dtype=torch.float32).reshape(-1,4)
    gt_labels=torch.tensor([b['label'] for b in regular]);used=set();tp=fp=0
    iou=box_iou(boxes,gt_boxes) if len(gt_boxes) else torch.empty((len(boxes),0))
    for i in range(len(boxes)):
        candidates=[j for j in range(len(gt_boxes)) if j not in used and gt_labels[j]==labels[i] and iou[i,j]>=iou_threshold]
        if candidates:
            j=max(candidates,key=lambda j:float(iou[i,j]));used.add(j);tp+=1
        else:
            # Region crowd/ignore tak dihitung sebagai FP ketika pusat prediksi masuk region.
            cx,cy=((boxes[i,:2]+boxes[i,2:])/2).tolist()
            in_ignore=any(b['label']==int(labels[i]) and b['box'][0]<=cx<=b['box'][2] and b['box'][1]<=cy<=b['box'][3] for b in ignored)
            if not in_ignore:fp+=1
    fn=len(gt_boxes)-len(used)
    return np.array([tp,fp,fn],np.int64)

def detection_confusion(matrix,pred,gt,threshold):
    regular=[b for b in gt if not b.get('ignore')];select=pred['scores']>=threshold
    boxes=pred['boxes'][select];labels=pred['labels'][select]
    g=torch.tensor([b['box'] for b in regular],dtype=torch.float32).reshape(-1,4)
    pairs=[]
    if len(boxes) and len(g):
        overlap=box_iou(boxes,g)
        for i,j in (overlap>=0.5).nonzero().tolist():pairs.append((float(overlap[i,j]),i,j))
    used_p=set();used_g=set()
    for _,i,j in sorted(pairs,reverse=True):
        if i in used_p or j in used_g:continue
        used_p.add(i);used_g.add(j);matrix[regular[j]['label'],int(labels[i])]+=1
    for i in range(len(boxes)):
        if i not in used_p:matrix[-1,int(labels[i])]+=1
    for j in range(len(g)):
        if j not in used_g:matrix[regular[j]['label'],-1]+=1

def coco_metrics(dataset,predictions,img_ids=None):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    ids=img_ids if img_ids is not None else [r['id'] for r in dataset['images']]
    if not ids:return {'map50':None,'map50_95':None,'status':'ground_truth_unavailable'}
    id_set=set(ids)
    if not any(a['image_id'] in id_set and not a.get('iscrowd',0) for a in dataset['annotations']):
        return {'map50':None,'map50_95':None,'status':'no_positive_ground_truth'}
    if not predictions:return {'map50':0.0,'map50_95':0.0,'status':'evaluated_no_predictions'}
    with contextlib.redirect_stdout(io.StringIO()):
        gt=COCO();gt.dataset=dataset;gt.createIndex();pd=gt.loadRes(predictions)
        evaluator=COCOeval(gt,pd,'bbox');evaluator.params.imgIds=ids;evaluator.params.maxDets=[1,10,100]
        evaluator.evaluate();evaluator.accumulate();evaluator.summarize()
    return {'map50':float(evaluator.stats[1]) if evaluator.stats[1]>=0 else None,
            'map50_95':float(evaluator.stats[0]) if evaluator.stats[0]>=0 else None,'status':'evaluated_COCO_style'}

@torch.no_grad()
def evaluate(model,samples,cfg,device,split='val'):
    selected=[s for s in samples if s['split']==split and any(s[t] is not None for t in TASKS)]
    root=Path(cfg.drive)/'evaluation'/split;root.mkdir(parents=True,exist_ok=True)
    if not selected:
        result={'split':split,'status':'no_accessible_ground_truth','metrics':None}
        write_json(root/'metrics.json',result);print(result);return result
    ds=BDDDataset(selected,cfg,False);model.eval();model.detector.score_thresh=cfg.map_score_threshold
    matrices={t:np.zeros((len(names),len(names)),np.int64) for t,names in DENSE_NAMES.items()}
    domain_matrices={};domain_det=defaultdict(lambda:np.zeros(3,np.int64));domain_ids=defaultdict(list);domain_support=defaultdict(int)
    det_cm=np.zeros((11,11),np.int64);det_count=np.zeros(3,np.int64);tolerances={r:np.zeros(4,np.int64) for r in (1,2,5)}
    support=defaultdict(int);errors=[]
    coco={'images':[],'annotations':[],'categories':[{'id':i,'name':n} for i,n in enumerate(DET_NAMES)],'info':{}}
    predictions=[]
    for images,targets,metadata in tqdm(make_loader(ds,cfg),desc=f'Evaluasi {split}, seluruh GT'):
        with autocast_context(cfg,device):output=model(images.to(device),predict=True)
        for i,meta in enumerate(metadata):
            s=meta['sample'];idx=meta['index'];pred=restore_boxes(output['detections'][i],meta['transform'])
            domains=[f'{field}={s["attributes"].get(field,"undefined")}' for field in ('weather','timeofday')]
            item={'id':s['id'],'key':s['key'],'split':split,'weather':s['attributes'].get('weather','undefined'),
                  'timeofday':s['attributes'].get('timeofday','undefined')}
            if s['det'] is not None:
                support['det']+=1
                counts=detection_counts(pred,s['det'],cfg.score_threshold);det_count+=counts
                detection_confusion(det_cm,pred,s['det'],cfg.score_threshold)
                item.update(det_tp=int(counts[0]),det_fp=int(counts[1]),det_fn=int(counts[2]))
                coco['images'].append({'id':idx,'height':s['height'],'width':s['width']})
                for b in s['det']:
                    x1,y1,x2,y2=b['box']
                    coco['annotations'].append({'id':len(coco['annotations'])+1,'image_id':idx,'category_id':b['label'],
                                               'bbox':[x1,y1,x2-x1,y2-y1],'area':(x2-x1)*(y2-y1),'iscrowd':int(b.get('ignore',False))})
                for box,score,label in zip(pred['boxes'].tolist(),pred['scores'].tolist(),pred['labels'].tolist()):
                    x1,y1,x2,y2=box;predictions.append({'image_id':idx,'category_id':label,'bbox':[x1,y1,x2-x1,y2-y1],'score':score})
                for d in domains:domain_det[d]+=counts;domain_ids[d].append(idx)
            for task in matrices:
                if s[task] is None:continue
                support[task]+=1
                gt=decode_mask(s[task]['path'],task,s[task]['encoding'])
                logits=unletterbox_logits(output['dense'][task][i],meta['transform']);seg_pred=logits.argmax(0).cpu().numpy().astype(np.uint8)
                confusion_update(matrices[task],gt,seg_pred)
                local=np.zeros_like(matrices[task]);confusion_update(local,gt,seg_pred)
                foreground=[1] if task=='lane' else [0,1] if task=='drive' else None
                item[task+'_iou']=dense_scores(local,foreground)['mean_iou']
                for d in domains:
                    key=(d,task)
                    if key not in domain_matrices:domain_matrices[key]=np.zeros_like(matrices[task])
                    domain_matrices[key]+=local;domain_support[key]+=1
                if task=='lane':
                    for r in tolerances:tolerances[r]+=lane_tolerance_counts(gt,seg_pred,r)
            errors.append(item)
    def pr(counts):
        tp,fp,fn=map(int,counts);return {'precision':tp/(tp+fp) if tp+fp else 0.0,'recall':tp/(tp+fn) if tp+fn else 0.0,'tp':tp,'fp':fp,'fn':fn}
    metrics={}
    if support['det']:
        metrics['det']={**pr(det_count),**coco_metrics(coco,predictions),'confusion':det_cm.tolist(),
                        'precision_recall_protocol':f'class-aware IoU>=0.5, confidence>={cfg.score_threshold}'}
    else:metrics['det']=None
    for task,cm in matrices.items():
        fg=[1] if task=='lane' else [0,1] if task=='drive' else None
        metrics[task]=dense_scores(cm,fg) if support[task] else None
        if task=='drive' and support[task]:metrics[task]['all_class_mean_iou']=dense_scores(cm)['mean_iou']
    if support['lane']:
        for r,(pm,npd,gm,ngt) in tolerances.items():
            p=pm/npd if npd else 0.0;rcl=gm/ngt if ngt else 0.0
            metrics['lane'][f'tolerance_{r}px']={'precision':float(p),'recall':float(rcl),'f1':float(2*p*rcl/(p+rcl)) if p+rcl else 0.0}
    domain_results={}
    for d in sorted(set(k[0] for k in domain_matrices)|set(domain_ids)):
        domain_results[d]={}
        if domain_ids[d]:domain_results[d]['det']={**pr(domain_det[d]),**coco_metrics(coco,predictions,domain_ids[d]),'images':len(domain_ids[d])}
        for task in matrices:
            if (d,task) in domain_matrices:
                fg=[1] if task=='lane' else [0,1] if task=='drive' else None
                domain_results[d][task]={**dense_scores(domain_matrices[(d,task)],fg),'images':domain_support[(d,task)]}
    result={'split':split,'status':'executed','support_images':dict(support),'metrics':metrics,'domains':domain_results,
            'protocol':{'native_resolution':True,'detection':'COCO style on accessible label version; not official BDD ignore-rule benchmark',
                        'lane':'binary all marking categories; strict pixel and tolerant pixel matching; not instance/polyline benchmark',
                        'drive':'mean IoU/Dice over direct+alternative; background included in confusion and FP counts',
                        'model_selection':split=='val','ground_truth_test':'only if physically available; no synthetic hold-out from train'}}
    write_json(root/'metrics.json',result);pd_frame=pd.DataFrame(errors);pd_frame.to_csv(root/'per_image_errors.csv',index=False)
    write_json(root/'coco_ground_truth.json',coco);write_json(root/'coco_predictions.json',predictions)
    # Ringkasan error terburuk diurutkan setelah evaluasi penuh.
    ranking=pd_frame.copy()
    columns=[c for c in ranking if c.endswith('_iou')]
    if columns:ranking['dense_mean_iou']=ranking[columns].mean(axis=1);ranking.sort_values('dense_mean_iou').to_csv(root/'worst_dense_errors.csv',index=False)
    if 'det_fn' in ranking:ranking.sort_values(['det_fn','det_fp'],ascending=False).to_csv(root/'worst_detection_errors.csv',index=False)
    print(json.dumps({'split':split,'support':dict(support),'detection':None if not metrics['det'] else {k:metrics['det'][k] for k in ('precision','recall','map50','map50_95')}},indent=2))
    return result

def plot_training(history,cfg):
    root=Path(cfg.drive)/'figures';root.mkdir(parents=True,exist_ok=True)
    fig,axes=plt.subplots(2,2,figsize=(13,8))
    records=[]
    for ax,task in zip(axes.ravel(),TASKS):
        epochs=[r['epoch'] for r in history]
        train=[np.nan if r['train'][task] is None else r['train'][task] for r in history]
        val=[np.nan if r['val'][task] is None else r['val'][task] for r in history]
        ax.plot(epochs,train,label='train (augmentasi aktif)');ax.plot(epochs,val,label='val');ax.set(title=task,xlabel='Epoch',ylabel='Raw task loss');ax.legend();ax.grid(alpha=0.3)
        if len(history)>=3:
            a=np.array(train[-3:]);b=np.array(val[-3:]);valid=np.isfinite(a)&np.isfinite(b)
            if valid.any():records.append({'task':task,'recent_train_loss':float(a[valid].mean()),'recent_val_loss':float(b[valid].mean()),
                                            'val_loss_slope_last3':float(np.polyfit(np.arange(len(b))[np.isfinite(b)],b[np.isfinite(b)],1)[0]) if np.isfinite(b).sum()>1 else None})
    fig.tight_layout();fig.savefig(root/'training_validation_curves.png',dpi=150);plt.show()
    write_json(Path(cfg.drive)/'reports/overfitting_loss_analysis.json',{'measurements':records,
        'interpretation':'Train dengan augmentasi berbeda distribusi dari val. Loss gap bukan bukti tunggal overfitting. Bandingkan juga evaluasi train tanpa augmentasi dan val dari checkpoint sama.'})

def plot_evaluation(result,cfg):
    if result.get('status')!='executed':return
    root=Path(cfg.drive)/'figures';root.mkdir(parents=True,exist_ok=True)
    for task,m in result['metrics'].items():
        if m is None:continue
        names=DET_NAMES+['background'] if task=='det' else DENSE_NAMES[task]
        cm=np.asarray(m['confusion'],np.float64);normal=cm/np.maximum(cm.sum(1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(10,9));image=ax.imshow(normal,cmap='Blues',vmin=0,vmax=1)
        ax.set(xticks=np.arange(len(names)),yticks=np.arange(len(names)),xticklabels=names,yticklabels=names,
               xlabel='Prediksi',ylabel='Ground truth',title=f'{result["split"]}: {task}, row normalized')
        plt.setp(ax.get_xticklabels(),rotation=90);fig.colorbar(image,ax=ax);fig.tight_layout()
        fig.savefig(root/f'{result["split"]}_{task}_confusion.png',dpi=140);plt.show()

def generalization_report(train_result,val_result,cfg):
    rows=[]
    if train_result.get('status')=='executed' and val_result.get('status')=='executed':
        for task in TASKS:
            a=train_result['metrics'].get(task);b=val_result['metrics'].get(task)
            metric='map50_95' if task=='det' else 'mean_iou'
            if a and b and a.get(metric) is not None and b.get(metric) is not None:
                rows.append({'task':task,'metric':metric,'train':a[metric],'val':b[metric],'gap':a[metric]-b[metric]})
    write_json(Path(cfg.drive)/'reports/generalization_gap.json',{'gaps':rows,
        'note':'Gap dihitung pada checkpoint sama dan tanpa augmentasi. Tidak menjamin bebas overfitting; validation telah dipakai model selection.'})
    return pd.DataFrame(rows)

PALETTE=np.array([[128,64,128],[244,35,232],[70,70,70],[102,102,156],[190,153,153],[153,153,153],
                  [250,170,30],[220,220,0],[107,142,35],[152,251,152],[70,130,180],[220,20,60],
                  [255,0,0],[0,0,142],[0,0,70],[0,60,100],[0,80,100],[0,0,230],[119,11,32]],np.uint8)

def combined_overlay(rgb,masks,pred,threshold):
    image=rgb.astype(np.float32);image=0.65*image+0.35*PALETTE[masks['sem']]
    for cls,color in [(0,np.array([0,220,80])),(1,np.array([0,150,255]))]:
        where=masks['drive']==cls;image[where]=0.55*image[where]+0.45*color
    image[masks['lane']==1]=[255,255,0];image=np.clip(image,0,255).astype(np.uint8)
    for b,score,label in zip(pred['boxes'],pred['scores'],pred['labels']):
        if score<threshold:continue
        x1,y1,x2,y2=map(int,b);cv2.rectangle(image,(x1,y1),(x2,y2),(255,80,80),2)
        cv2.putText(image,f'{DET_NAMES[int(label)]} {float(score):.2f}',(x1,max(y1-4,12)),cv2.FONT_HERSHEY_SIMPLEX,0.4,(255,255,255),1,cv2.LINE_AA)
    return image

@torch.no_grad()
def infer_one(model,path,cfg,device):
    with Image.open(path) as im:rgb=np.array(im.convert('RGB'))
    sample={'path':str(path),'id':Path(path).stem,'key':'single','split':'external','det':None,'sem':None,'lane':None,'drive':None}
    image,_,meta=BDDDataset([sample],cfg)[0]
    model.eval();model.detector.score_thresh=cfg.map_score_threshold
    with autocast_context(cfg,device):out=model(image[None].to(device),predict=True)
    pred=restore_boxes(out['detections'][0],meta['transform'])
    masks={t:unletterbox_logits(out['dense'][t][0],meta['transform']).argmax(0).cpu().numpy().astype(np.uint8) for t in ('sem','lane','drive')}
    return rgb,pred,masks,combined_overlay(rgb,masks,pred,cfg.score_threshold)

@torch.no_grad()
def infer_every_image(model,physical_images,cfg,device):
    """Setiap physical image valid, termasuk unlabeled, duplikat, unknown, quarantine."""
    root=Path(cfg.drive)/'predictions';root.mkdir(parents=True,exist_ok=True)
    # Per-file writes di Drive lambat; keluaran sengaja persisten agar dapat dilanjutkan.
    check_disk(root,5)
    checkpoint=Path(cfg.drive)/'checkpoints/best.pt'
    if not checkpoint.exists():raise FileNotFoundError('Tidak menginferensi dengan head yang belum dilatih')
    model_hash=sha_file(checkpoint)
    contract={'checkpoint_sha256':model_hash,'height':cfg.height,'width':cfg.width,
              'map_score_threshold':cfg.map_score_threshold,'overlay_threshold':cfg.score_threshold,'nms':cfg.nms_threshold,
              'source_sha256':{name:sha_file(Path(__file__).with_name(name)) for name in ('bdd4_data.py','bdd4_model.py','bdd4_engine.py')},'torch':torch.__version__}
    run_file=root/'inference_contract.json'
    if run_file.exists() and json.loads(run_file.read_text())!=contract:
        raise RuntimeError('Prediksi lama memakai model/config berbeda. Pakai folder eksperimen baru.')
    write_json(run_file,contract)
    rows=physical_images.to_dict('records');success=[];failures=[]
    def persist_progress():
        with (root/'prediction_index.jsonl').open('w') as index_file:
            for item in success:index_file.write(json.dumps(item)+'\n')
        write_json(root/'failed_images.json',failures)
        write_json(root/'partial_progress.json',{'completed':len(success),'expected_valid':int(physical_images.valid.sum()),'finalized':False})
    for row in tqdm(rows,desc='Inferensi SEMUA physical images'):
        if not row.get('valid',False):
            failures.append({'path':row['path'],'stage':'integrity_audit','error':row.get('error','invalid_image')});continue
        # Key stabil terhadap perubahan lokasi cache runtime Colab.
        key=hashlib.sha256((row['relative']+'|'+row['sha256']).encode()).hexdigest()[:24]
        folder=root/key[:2]/key;folder.mkdir(parents=True,exist_ok=True)
        done=folder/'record.json'
        required=[folder/(t+'.png') for t in ('sem','lane','drive')]+[folder/'overlay.jpg',folder/'boxes.json']
        if done.exists() and all(p.exists() for p in required):
            try:
                record=json.loads(done.read_text())
                if record['source_sha256']!=row['sha256'] or record['relative']!=row['relative']:raise ValueError('Source mismatch')
                json.loads((folder/'boxes.json').read_text())
                with Image.open(folder/'overlay.jpg') as overlay_image:overlay_image.load()
                # Periksa ukuran dan decode; file yang terputus tidak dianggap selesai.
                for path in required[:3]:
                    with Image.open(path) as im:
                        im.load()
                        if im.size!=(int(row['width']),int(row['height'])):raise ValueError('size mismatch')
                success.append(record);continue
            except Exception:done.unlink(missing_ok=True)
        try:
            rgb,pred,masks,overlay=infer_one(model,row['path'],cfg,device)
            for t,mask in masks.items():
                temp=folder/(t+'.tmp.png');Image.fromarray(mask).save(temp);temp.replace(folder/(t+'.png'))
            boxes={k:v.tolist() for k,v in pred.items()};boxes.update(class_names=DET_NAMES,coordinates='xyxy original pixels',label_index_base=0)
            write_json(folder/'boxes.json',boxes)
            temp=folder/'overlay.tmp.jpg';Image.fromarray(overlay).save(temp,quality=85);temp.replace(folder/'overlay.jpg')
            record={'key':key,'source':row['path'],'relative':row['relative'],'source_sha256':row['sha256'],
                    'split':row['split'],'shape':[int(row['height']),int(row['width'])],'folder':str(folder),
                    'checkpoint_sha256':model_hash,'outputs':['boxes.json','sem.png','lane.png','drive.png','overlay.jpg']}
            write_json(done,record);success.append(record)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache();failures.append({'path':row['path'],'stage':'inference','error':'CUDA_OOM'})
        except Exception as e:failures.append({'path':row['path'],'stage':'inference','error':type(e).__name__,'detail':str(e)[:300]})
        if len(success)%500==0:
            persist_progress()
            # Tambahan disk diperiksa berkala; tidak memotong jumlah gambar.
            if shutil.disk_usage(root).free<512*1024**2:raise RuntimeError('Disk output hampir penuh. Tambah ruang dan lanjutkan; tidak ada sampling otomatis.')
    with (root/'prediction_index.jsonl').open('w') as f:
        for row in success:f.write(json.dumps(row)+'\n')
    write_json(root/'failed_images.json',failures)
    valid_expected=int(physical_images.valid.sum())
    report={'accessible_physical_images':len(rows),'valid_images_expected':valid_expected,'successful_predictions':len(success),
            'invalid_images':len(rows)-valid_expected,'inference_failures':sum(f['stage']=='inference' for f in failures),
            'all_valid_images_completed':len(success)==valid_expected,'duplicate_policy':'each physical occurrence has an output',
            'output_contract':contract}
    write_json(root/'inference_coverage.json',report)
    if len(success)!=valid_expected:raise RuntimeError('Inferensi belum lengkap. Lihat failed_images.json, perbaiki, lalu jalankan ulang sel ini.')
    print(json.dumps(report,indent=2));return success,report

def export_model(model,cfg,device):
    destination=Path(cfg.drive)/'export';destination.mkdir(parents=True,exist_ok=True)
    model.eval();wrapper=ExportRaw(model).to(device).eval()
    example=torch.zeros(1,3,cfg.height,cfg.width,device=device)
    with torch.no_grad():
        reference=wrapper(example)
        traced=torch.jit.trace(wrapper,example,strict=False,check_trace=False)
        actual=traced(example)
        for a,b in zip(reference,actual):torch.testing.assert_close(a,b,rtol=1e-4,atol=1e-5)
        random_input=torch.rand_like(example)
        for a,b in zip(wrapper(random_input),traced(random_input)):torch.testing.assert_close(a,b,rtol=1e-4,atol=1e-5)
    traced.save(str(destination/'bdd4_raw_fixed_shape.ts'))
    # State dict lengkap: reconstruction menggunakan FourTaskNet(pretrained=False).
    atomic_checkpoint(destination/'model_state.pt',{'model':model.state_dict(),'config':cfg.dump(),'class_names':{'det':DET_NAMES,**DENSE_NAMES}})
    contract={'format':'TorchScript raw outputs + PyTorch state dict','input':[1,3,cfg.height,cfg.width],
              'input_range':'RGB float [0,1], letterboxed; ImageNet normalization is inside model',
              'outputs':['cls_logits [B,A,10]','box_distances [B,A,4]','centerness_logits [B,A,1]','anchors [B,A,4]',
                         'semantic_logits [B,19,H,W]','lane_logits [B,2,H,W]','drivable_logits [B,3,H,W]'],
              'postprocessing':'FCOS BoxLinearCoder(normalize_by_size=True), sqrt(sigmoid(class)*sigmoid(centerness)), per-class NMS, inverse letterbox; see bdd4_model.py and infer_one',
              'limitations':'Fixed input shape and batch=1 for traced graph; raw graph excludes NMS and resizing to original resolution. End-to-end inference remains available with supporting Python modules.',
              'verification':'raw eager vs TorchScript all outputs checked in export cell; not ONNX/TensorRT verification'}
    write_json(destination/'export_contract.json',contract);return destination

def gpu_memory_probe(model,cfg,device):
    """Probe kapasitas dengan tensor sintetis; tidak mengubah bobot atau melaporkan akurasi."""
    if device.type!='cuda':raise RuntimeError('Training notebook memerlukan Colab GPU: Runtime > Change runtime type > GPU.')
    saved_rng=rng_state();model.train();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)
    images=torch.rand(cfg.batch_size,3,cfg.height,cfg.width,device=device)
    targets=[]
    for _ in range(cfg.batch_size):
        targets.append({'boxes':torch.tensor([[10,10,cfg.width-10,cfg.height-10]],device=device,dtype=torch.float32),
                        'labels':torch.tensor([2],device=device),'ignore_boxes':torch.empty((0,4),device=device),'det_valid':True,
                        **{t:torch.zeros((cfg.height,cfg.width),device=device,dtype=torch.long) for t in ('sem','lane','drive')}})
    criterion=MultiTaskCriterion({'sem':[1]*19,'lane':[1]*2,'drive':[1]*3}).to(device)
    try:
        with autocast_context(cfg,device):
            output=model(images,targets);loss,_,_=criterion(model,output,targets)
        loss.backward();torch.cuda.synchronize()
        peak=torch.cuda.max_memory_allocated(device)/1024**3
        total=torch.cuda.get_device_properties(device).total_memory/1024**3
        # AdamW menyimpan dua momentum fp32; grad dan parameter telah tercakup dalam peak.
        optimizer_reserve=sum(p.numel()*p.element_size()*2 for p in model.parameters())/1024**3
        report={'gpu':torch.cuda.get_device_name(device),'peak_probe_gib':peak,'gpu_total_gib':total,
                'estimated_adam_states_gib':optimizer_reserve,'batch_size':cfg.batch_size,'input':[cfg.height,cfg.width],
                'synthetic_capacity_test_only':True}
        if peak+optimizer_reserve>0.9*total:raise RuntimeError('Headroom GPU terlalu kecil; buat config batch lebih kecil sebelum training.')
        write_json(Path(cfg.work)/'reports/gpu_memory_probe.json',report);print(report);return report
    finally:
        model.zero_grad(set_to_none=True);del images,targets,criterion
        restore_rng(saved_rng);torch.cuda.empty_cache()

def visualize_errors(model,samples,cfg,device,split='val',number=6):
    path=Path(cfg.drive)/'evaluation'/split/'worst_dense_errors.csv'
    if not path.exists():return
    rows=pd.read_csv(path);by_key={str(s['key']):s for s in samples}
    selected=[]
    for _,row in rows.iterrows():
        s=by_key.get(str(row['key']))
        if s is not None:selected.append(s)
        if len(selected)==number:break
    for s in selected:
        rgb,pred,masks,overlay=infer_one(model,s['path'],cfg,device)
        fig,axes=plt.subplots(2,4,figsize=(18,8))
        ground=rgb.copy()
        for b in s.get('det') or []:
            x1,y1,x2,y2=map(int,b['box']);cv2.rectangle(ground,(x1,y1),(x2,y2),(0,255,0),2)
        axes[0,0].imshow(ground);axes[0,0].set_title('GT bounding boxes bila tersedia')
        axes[1,0].imshow(overlay);axes[1,0].set_title('Prediksi empat tugas')
        for col,task in enumerate(('sem','lane','drive'),1):
            axes[0,col].imshow(masks[task],vmin=0,vmax=len(DENSE_NAMES[task])-1,cmap='tab20');axes[0,col].set_title(task+' prediksi')
            if s[task] is None:axes[1,col].text(0.1,0.5,'GT tidak tersedia')
            else:
                gt=decode_mask(s[task]['path'],task,s[task]['encoding']);axes[1,col].imshow(np.ma.masked_equal(gt,255),vmin=0,vmax=len(DENSE_NAMES[task])-1,cmap='tab20')
            axes[1,col].set_title(task+' ground truth')
        for ax in axes.ravel():ax.axis('off')
        fig.suptitle(f'{split} — {s["id"]} — {s["attributes"]}');fig.tight_layout()
        root=Path(cfg.drive)/'figures/errors';root.mkdir(parents=True,exist_ok=True)
        fig.savefig(root/(s['id']+'.jpg'),dpi=120);plt.show()


In [ ]:
sys.path.insert(0,'/content/bdd4_lib')
import json, shutil, platform, datetime, random, math, hashlib, zipfile, sqlite3
from collections import Counter
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from bdd4_data import *
from bdd4_model import *
from bdd4_engine import *

cfg=Config()
# Parameter eksperimen dapat diubah di sini SEBELUM audit/training.
# cfg.height, cfg.width = 512, 896
# cfg.batch_size, cfg.accumulation = 1, 8
# cfg.epochs = 50
# cfg.drive='/content/drive/MyDrive/BDD4_research_experiment_02'
assert cfg.height%32==0 and cfg.width%32==0
seed_everything(cfg.seed)
assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > GPU.'
device=torch.device('cuda')
print('GPU:',torch.cuda.get_device_name(0))
print('GPU memory GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print('RAM GiB:',round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1024**3,2))
print('PyTorch:',torch.__version__,'torchvision:',importlib.metadata.version('torchvision'))
check_disk('/content',30)  # ZIP + hasil ekstrak + cache/proses; prediksi besar membutuhkan ruang terpisah.


In [ ]:
from google.colab import drive
# Mount jarak jauh di Colab; tidak memindahkan dataset ke komputer lokal.
drive.mount('/content/drive')
Path(cfg.drive).mkdir(parents=True,exist_ok=True)
check_disk(cfg.drive,3)  # checkpoint, laporan, dan ekspor; estimasi keluaran penuh diperiksa di tahap 12.
quota_report={}
try:
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    credentials,_=google.auth.default()
    quota=build('drive','v3',credentials=credentials,cache_discovery=False).about().get(fields='storageQuota').execute()['storageQuota']
    quota_report={k:int(v) for k,v in quota.items()}
    if 'limit' in quota_report:
        quota_report['available_bytes']=quota_report['limit']-quota_report.get('usage',0)
        print('Kuota Drive tersisa GiB:',quota_report['available_bytes']/1024**3)
        assert quota_report['available_bytes']>=3*1024**3, 'Kuota Drive tidak cukup untuk checkpoint dan ekspor.'
except AssertionError: raise
except Exception as e:
    print('Kuota API Drive belum terverifikasi:',type(e).__name__,'. Periksa kapasitas akun Drive sebelum inferensi penuh.')
    quota_report={'status':'API_unverified','filesystem_available_bytes':shutil.disk_usage(cfg.drive).free}
write_json(Path(cfg.work)/'reports/drive_quota.json',quota_report)
write_json(Path(cfg.work)/'reports/config.json',cfg.dump())
environment={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'python':platform.python_version(),
             'torch':torch.__version__,'torchvision':importlib.metadata.version('torchvision'),
             'gpu':torch.cuda.get_device_name(0),'seed':cfg.seed,'source_sha256':{p.name:sha_file(p) for p in Path('/content/bdd4_lib').glob('*.py')}}
write_json(Path(cfg.work)/'reports/environment.json',environment)
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
(Path(cfg.work)/'reports/pip_freeze.txt').write_text(freeze)


## 2. Kaggle Dataset Download

Dataset diunduh langsung menggunakan **KaggleHub**, dipatok ke versi 2. Cache berada di disk Colab. Dataset publik dicoba tanpa kredensial terlebih dahulu; jika Kaggle mewajibkan login, autentikasi berlangsung di Colab. Tidak perlu mengunduh `kaggle.json` ke laptop.

Audit jarak jauh saat penyusunan menemukan 136.002 file, total ekstraksi 9.884.138.351 byte: 100K images, 10K images, salinan 10K segmentation images, 8K mask ID, 8K colormap, serta dua JSON train/val. Ini **observasi struktur**, bukan hasil decode/integritas menyeluruh. Audit Colab di tahap berikut tetap menjadi sumber kebenaran.

Sumber dataset: [Kaggle BDD100K](https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k). API: [KaggleHub resmi](https://github.com/Kaggle/kagglehub).

In [ ]:
os.environ['KAGGLEHUB_CACHE']=str(Path(cfg.work)/'kaggle_cache')
import kagglehub
# Opsional: simpan KAGGLE_USERNAME/KAGGLE_KEY di Colab Secrets; nilainya tidak dicetak.
try:
    from google.colab import userdata
    for name in ('KAGGLE_USERNAME','KAGGLE_KEY'):
        try: os.environ[name]=userdata.get(name)
        except Exception: pass
except ImportError: pass
try:
    kaggle_root=Path(kagglehub.dataset_download(cfg.kaggle_handle))
except Exception as first_error:
    print('Unduhan publik belum berhasil:',type(first_error).__name__)
    print('Login Kaggle di Colab, lalu mencoba ulang versi yang sama.')
    kagglehub.login()
    kaggle_root=Path(kagglehub.dataset_download(cfg.kaggle_handle))
print('Lokasi data di Colab:',kaggle_root)
report_dir=Path(cfg.work)/'reports';report_dir.mkdir(parents=True,exist_ok=True)
assets=inventory([kaggle_root],report_dir)
roots=[kaggle_root]


## 3. Dataset Integrity Audit

Semua gambar fisik didekode penuh, dihitung SHA-256 byte dan piksel RGB, lalu dikelompokkan berdasarkan ID dan isi. Pemeriksaan konservatif near-duplicate menggunakan pHash Hamming ≤4 dan RMSE thumbnail ≤1,5. Kandidat dengan piksel berbeda akan dikarantina untuk supervised learning; semua gambar valid tetap masuk inferensi.

**Pencegahan leakage:** `trainA`, `trainB`, `testA`, `testB` adalah subfolder tambahan pada mirror ini, bukan split benchmark. Split dibaca dari komponen persis `train`, `val`, atau `test` yang terluar. Kelompok lintas split diprioritaskan `test > val > train`; label dari split yang lebih rendah tidak dipindahkan menjadi target held-out. Konflik video yang teridentifikasi juga dikarantina. Pengeluaran ini wajib terlihat di laporan, sehingga cakupan train dapat berkurang bila mirror mempunyai kontaminasi.

Hash dan pHash tidak menjamin menemukan semua kemiripan antar-frame. Audit mendokumentasikan batas tersebut dan tidak menjanjikan bebas leakage untuk scene yang asal videonya tidak diketahui. Sel ini tidak membuang file asli.

In [ ]:
physical_images,valid_images=audit_images(assets,report_dir,workers=max(1,min(4,os.cpu_count() or 2)))
print('Gambar fisik:',len(physical_images),'valid:',len(valid_images),'gagal decode:',int((~physical_images.valid).sum()))
print('Kelompok unik setelah deduplikasi:',valid_images.group.nunique())
display(physical_images.groupby(['split','valid']).size().rename('images').reset_index())
# Tampilkan kegagalan secara eksplisit. Tidak menganggap file korup sebagai gambar tanpa objek.
display(physical_images.loc[~physical_images.valid,['path','error']] if 'error' in physical_images else pd.DataFrame())


## 4. Exploratory Data Analysis

EDA menggunakan seluruh inventaris dan seluruh record JSON, tanpa sampling tersembunyi. Colormap hanya untuk visualisasi; target semantic berasal dari mask ID single-channel. Histogram kondisi di bawah mengukur record anotasi yang memiliki metadata, sehingga tidak boleh digeneralisasi sebagai metadata semua gambar test.

Pembagian resmi yang diharapkan adalah 70K/10K/20K untuk 100K images dan 7K/1K/2K untuk 10K images. Paket 10K bukan subset sempurna dari 100K; overlap diatasi berdasarkan identitas gambar, bukan asumsi jumlah.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
physical_images[physical_images.valid].groupby('split').size().plot.bar(ax=axes[0],title='Gambar fisik per split (termasuk salinan)')
valid_images.groupby(['width','height']).size().sort_values().plot.barh(ax=axes[1],title='Resolusi seluruh gambar valid')
plt.tight_layout();plt.show()
eda={'weather':Counter(),'timeofday':Counter(),'scene':Counter(),'category':Counter(),'record_count':0}
json_paths=assets.loc[assets.kind=='det_json','path'].tolist()
for path in json_paths:
    for record in iter_records(path):
        eda['record_count']+=1
        for field in ('weather','timeofday','scene'):eda[field][(record.get('attributes') or {}).get(field,'undefined')]+=1
        for label in record_labels(record) or []:eda['category'][label.get('category','undefined')]+=1
fig,axes=plt.subplots(1,3,figsize=(17,4))
for ax,field in zip(axes,('weather','timeofday','scene')):
    pd.Series(eda[field]).sort_values().plot.barh(ax=ax,title=field)
plt.tight_layout();plt.show()
write_json(report_dir/'eda.json',{k:dict(v) if isinstance(v,Counter) else v for k,v in eda.items()})
print('Record JSON yang dibaca:',eda['record_count'])


## 5. Annotation Processing

Target yang digunakan:

| Tugas | Sumber utama | Encoding |
|---|---|---|
| Deteksi | JSON legacy Kaggle 2018 | 10 kelas; alias person→pedestrian, motor→motorcycle, bike→bicycle |
| Semantic | `seg/labels/*/*_train_id.png` atau mask resmi `sem_seg/masks` | 0–18; 255 ignore |
| Lane | Mask lane resmi bila tersedia; selain itu poligon `lane` pada JSON Kaggle | biner semua marking; bit background resmi dibaca dari **bit 3**, bukan `raw > 0` |
| Drivable | Mask resmi bila tersedia; selain itu poligon `drivable area` | 0 direct, 1 alternative, 2 background; 255 ignore |

Encoding lane diperiksa terhadap [kode evaluator resmi](https://github.com/bdd100k/bdd100k/blob/9ac17c6c7c51d2fc83065fccd707cd5b1882a293/bdd100k/eval/lane.py). Angka 255 pada aset lane berarti background dan tidak diperlakukan seperti ignore semantic.

**Retrieval otomatis:** aset mask lane/drivable yang tidak ada dicoba dari lokasi unduhan yang dirujuk [dokumentasi BDD100K](https://github.com/bdd100k/bdd100k/blob/9ac17c6c7c51d2fc83065fccd707cd5b1882a293/doc/source/download.rst), dengan checksum MD5 rilis. Server resmi ETH tidak dapat dijangkau dari lingkungan penyusunan pada 13 November 2026; notebook tidak menganggap link itu selalu hidup. Kegagalan dicatat, lalu anotasi poligon asli yang tersedia tetap dapat dipakai.

Jika perlu URL portal resmi terbaru, isi `cfg.official_urls={'lane':['URL_RESMI'], 'drive':['URL_RESMI'], 'sem':['URL_RESMI']}` dan jalankan ulang tahap 5–6. Verifikasi checksum dan inventaris wajib lolos. Jika aset wajib benar-benar tidak tersedia, training empat tugas akan berhenti; tidak ada pseudo-label.

Rasterisasi poligon mengikuti bentuk cubic Bezier L/C, urutan overlap, dan gaya garis 1 point pada dpi=100 dari renderer Scalabel/BDD100K. [Referensi transformasi Scalabel](https://github.com/scalabel/scalabel/blob/master/scalabel/label/transforms.py). Ini transformasi anotasi manusia, bukan prediksi model. **Frame tanpa mask dan tanpa poligon suatu tugas ditandai missing**; tidak diciptakan mask background. Karena konversi dan versi label memengaruhi hasil, metrik poligon rasterisasi tidak diklaim setara benchmark mask resmi.

In [ ]:
expected_labeled_counts={'sem':{'train':7000,'val':1000},'lane':{'train':70000,'val':10000},'drive':{'train':70000,'val':10000}}
missing_mask_tasks=[]
for task,expected in expected_labeled_counts.items():
    for split,count in expected.items():
        present={image_id(p) for p in assets.loc[(assets.kind==task)&(assets.split==split),'path']}
        if len(present)<count:
            missing_mask_tasks.append(task);break
print('Paket mask yang belum lengkap relatif terhadap rilis train/val:',missing_mask_tasks)
if missing_mask_tasks and cfg.download_official:
    official_root=retrieve_official(missing_mask_tasks,cfg)
    roots=[kaggle_root,official_root]
    assets=inventory(roots,report_dir)
    # Retrieval mask tidak menambahkan gambar; bila sumber baru ternyata berisi gambar, audit semuanya ulang.
    if set(assets.loc[assets.kind=='image','path'])!=set(physical_images.path):
        physical_images,valid_images=audit_images(assets,report_dir,workers=max(1,min(4,os.cpu_count() or 2)))
mask_lookup=audit_masks(assets,report_dir)
json_paths=assets.loc[assets.kind=='det_json','path'].tolist()
annotation_db=build_annotation_db(json_paths,Path(cfg.work)/'annotations.sqlite',report_dir/'annotation_audit.json')
write_json(report_dir/'annotation_file_hashes.json',{str(Path(p).relative_to(kaggle_root)):sha_file(p) for p in json_paths if Path(p).is_relative_to(kaggle_root)})
print('Database anotasi:',annotation_db)


In [ ]:
# Cache mask hasil rasterisasi dipatok ke hash source, agar resume Colab tidak harus selalu merender ulang.
raster_version=sha_file(Path('/content/bdd4_lib/bdd4_data.py'))[:16]
mask_cache=Path(cfg.drive)/f'generated_masks_{raster_version}.zip'
if mask_cache.exists():
    print('Memulihkan cache rasterisasi dari Drive')
    safe_unzip(mask_cache,cfg.work)


## 6. Multi-task Dataset Preparation

Manifest menggabungkan label hanya untuk gambar yang identitasnya telah diverifikasi. Boolean ketersediaan per tugas terpisah: missing detection tidak menjadi negative detection; missing dense mask diisi 255 untuk loss masking. **Gambar berlabel deteksi dengan nol objek tetap negative yang valid** bila record anotasinya tersedia.

Semua canonical training images dengan setidaknya satu label masuk setiap epoch, tanpa `drop_last`. Assertion di training memverifikasi jumlah unique image yang telah dipakai. Penggabungan duplikat mempertahankan label berbagai tugas pada satu gambar; konflik label/mask tidak dipilih sembarangan. Report mencatat file korup, konflik, missing label, deduplikasi, dan pengecualian lintas split.

In [ ]:
samples=prepare_manifest(valid_images,mask_lookup,annotation_db,cfg)
coverage=coverage_report(physical_images,valid_images,samples,cfg)
fingerprint=dataset_fingerprint(samples,assets,cfg)
write_json(report_dir/'dataset_fingerprint.json',{'sha256':fingerprint,'kaggle_version':cfg.kaggle_handle})
statistics=training_statistics(samples,cfg)  # HANYA train, bukan val/test.
print('Fingerprint:',fingerprint)
print('Deteksi class counts:',dict(zip(DET_NAMES,statistics['det_box_counts'])))
# Semua split tetap dapat ditelusuri sebelum training dimulai.
for task in TASKS:
    print(task,{split:sum(s['split']==split and s[task] is not None for s in samples) for split in SPLITS})


In [ ]:
# Simpan cache dan provenance untuk pemulihan lintas runtime.
generated=Path(cfg.work)/'generated_masks'
if generated.exists() and not mask_cache.exists():
    tmp=Path(cfg.work)/'generated_masks.zip'
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as archive:
        for path in generated.rglob('*.png'):archive.write(path,path.relative_to(cfg.work))
    shutil.copy2(tmp,mask_cache);tmp.unlink()
persistent_reports=Path(cfg.drive)/'reports';persistent_reports.mkdir(parents=True,exist_ok=True)
for path in report_dir.iterdir():
    if path.is_file():shutil.copy2(path,persistent_reports/path.name)
shutil.copy2(Path(cfg.work)/'manifest.json',Path(cfg.drive)/'manifest.json')
source_dir=Path(cfg.drive)/'source';source_dir.mkdir(exist_ok=True)
for path in Path('/content/bdd4_lib').glob('*.py'):shutil.copy2(path,source_dir/path.name)


## 7. Model Architecture

**Pilihan:** ResNet-50 + FPN pretrained COCO, head anchor-free FCOS, serta tiga decoder dense terpisah. Satu pass backbone menghasilkan C2–C5. FCOS memakai P3–P7; decoder dense menerima P2–P5 agar fitur stride 4 membantu struktur tipis. Setiap decoder mempunyai proyeksi multi-skala, fusion, GroupNorm, dropout, dan prediksi per piksel.

| Pendekatan | Cakupan dan pertimbangan |
|---|---|
| YOLOP asli | Deteksi + drivable + lane, tiga tugas; belum mempunyai semantic head 19 kelas |
| Empat model terpisah | Valid sebagai baseline ablation, tetapi menghitung backbone berulang |
| BDD4 yang diimplementasikan | Satu backbone dan empat head; reuse bobot detector matang, mudah diaudit untuk label parsial |

Model ini merupakan rancangan eksperimen, **bukan reproduksi YOLOP dan bukan klaim state of the art**. COCO backbone/FPN/regression/centerness ditransfer. Logit kelas yang cocok diinisialisasi dari COCO; rider memakai person dan traffic sign memakai stop sign hanya sebagai initialisasi. Tiga dense decoder dan jalur C2 baru harus dilatih pada BDD100K. Frozen BatchNorm dan GroupNorm dipilih untuk batch kecil.

Sumber: [FCOS torchvision](https://docs.pytorch.org/vision/0.24/models/fcos.html), [YOLOP resmi](https://github.com/hustvl/YOLOP), [BDD100K heterogeneous multitask learning](https://arxiv.org/abs/1805.04687).

In [ ]:
resume_available=(Path(cfg.drive)/'checkpoints/last.pt').exists()
model=FourTaskNet(pretrained=not resume_available,max_detections=cfg.max_detections).to(device)
model.detector.nms_thresh=cfg.nms_threshold
print('Parameter trainable:',f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print('Empat tugas:',TASKS)
print('COCO initialization digunakan pada eksperimen baru; resume akan memuat bobot checkpoint.')


## 8. Training Configuration

Loss deteksi terdiri atas focal classification, generalized IoU, dan centerness BCE. Region crowd/ignore tidak dihukum sebagai negative classification. Dense tasks memakai class-weighted cross entropy + Dice; class weights dihitung dari piksel train saja. Lane Dice berfokus pada foreground.

Loss tugas yang tidak berlabel **tidak ikut menjumlahkan loss ataupun regularizer balancing**. Bobot adaptif menggunakan formulasi praktis homoscedastic uncertainty: `exp(-s_t) × L_t + s_t`, dengan log-variance dibatasi [-3,3]. Ini adaptasi untuk loss gabungan; validitas probabilistik persis dari semua komponen tidak diklaim. Monitor raw loss dan bobotnya. [Kendall et al.](https://arxiv.org/abs/1705.07115).

AdamW memakai learning rate backbone 0,1× head, warm-up + cosine decay, AMP FP16/BF16, gradient accumulation, gradient clipping, dan early stopping. Jika gradien FP16 overflow, scale diturunkan dan seluruh window akumulasi diulang dengan RNG yang sama; gambar tidak dibuang. Tidak ada augmentasi yang mengubah semantik arah vertikal: hanya horizontal flip yang sinkron untuk box/mask serta jitter fotometrik. Resize mempertahankan aspect ratio melalui letterbox; padding diabaikan pada dense loss. Lane target di-downsample dengan occupancy agar garis tipis tidak hilang; evaluasi tetap pada resolusi asli.

Pemilihan best checkpoint menggunakan mean `log(1+val_loss/initial_val_loss)` dari tugas yang tersedia, bukan loss berbobot uncertainty yang dapat bergeser. Validation digunakan untuk selection; test tidak digunakan untuk pemilihan model, threshold, class weight, atau optimizer.

In [ ]:
gpu_probe=gpu_memory_probe(model,cfg,device)
write_json(Path(cfg.drive)/'reports/gpu_memory_probe.json',gpu_probe)
print('Effective batch maksimal:',cfg.batch_size*cfg.accumulation)
print('Epoch target:',cfg.epochs,'early stopping patience:',cfg.patience)
print('Checkpoint setiap',cfg.checkpoint_steps,'optimizer steps, plus akhir epoch dan model terbaik.')
# Catatan: jika gagal OOM, gunakan batch lebih kecil pada config dan folder eksperimen baru.
# Jangan menurunkan resolusi/men-skip batch secara otomatis di tengah eksperimen.


## 9. Multi-task Training

Sel ini benar-benar melatih semua gambar train berlabel. Urutan shuffle deterministik per epoch; augmentasi ditentukan seed + epoch + image key. Checkpoint mencakup bobot, optimizer, scheduler, scaler, RNG, cursor batch, akumulasi statistik epoch, dan daftar indeks yang telah digunakan. Resume dimulai dari optimizer boundary terakhir yang tersimpan; maksimum beberapa step setelah checkpoint mungkin perlu diulang bila sesi putus.

Checkpoint ditulis atomik ke Google Drive. Konfigurasi, fingerprint dataset, versi PyTorch, dan hash source harus cocok. Setelah training/early stopping, bobot best validation dimuat untuk seluruh evaluasi dan inferensi. Durasi penuh tidak dijanjikan selesai dalam kuota satu sesi Colab.

In [ ]:
model,history=train(model,samples,statistics,cfg,fingerprint,device,resume=True)
print('Training epoch selesai:',len(history))
print('Best checkpoint:',Path(cfg.drive)/'checkpoints/best.pt')
plot_training(history,cfg)


## 10. Model Evaluation

Evaluasi memproses seluruh GT valid pada split yang diminta. Box dikembalikan ke koordinat asli. Logit dense di-crop dari letterbox, di-upsample ke ukuran asli, lalu di-argmax; ignore pixel tidak masuk confusion matrix.

- **Deteksi:** precision/recall micro pada confidence 0,25 dan IoU 0,5 yang ditetapkan sebelumnya; mAP@50 dan mAP@50:95 COCO-style dari prediksi threshold rendah 0,001, NMS 0,6, maxDet 100. Versi label legacy dicatat; ini bukan angka benchmark Detection 2020 atau evaluator BDD resmi.
- **Semantic:** mean IoU, macro Dice, per-class IoU/Dice, confusion matrix 19 kelas; kelas tanpa union dilaporkan `null`, tidak dipaksa bernilai 1.
- **Lane:** precision, recall, F1, IoU foreground pada piksel asli; tambahan matching toleransi 1/2/5 piksel. Head ini segmentasi biner semua marking, bukan rekonstruksi lane instance/polynomial atau klasifikasi atribut lane resmi.
- **Drivable:** mean IoU dan Dice atas direct/alternative, dengan background tetap memengaruhi false positive dan confusion; mIoU semua tiga kelas juga dicatat.

Weather/lighting dibandingkan pada subset dengan metadata, lengkap dengan support gambar per tugas. Test hanya dievaluasi jika GT benar-benar tersedia secara fisik. Jika tidak, hasil `no_accessible_ground_truth`/`null` ditulis; validation tidak diganti namanya menjadi test.

In [ ]:
val_result=evaluate(model,samples,cfg,device,'val')
test_result=evaluate(model,samples,cfg,device,'test')
plot_evaluation(val_result,cfg)
if test_result.get('status')=='executed':plot_evaluation(test_result,cfg)
metric_rows=[]
for split,result in [('val',val_result),('test',test_result)]:
    if result.get('status')!='executed':continue
    for task,metrics in result['metrics'].items():
        if metrics is None:continue
        fields=('precision','recall','map50','map50_95') if task=='det' else ('mean_iou','dice','precision','recall','f1','iou')
        metric_rows.append({'split':split,'task':task,**{k:metrics[k] for k in fields if k in metrics}})
display(pd.DataFrame(metric_rows))


In [ ]:
# Perbandingan domain berasal dari seluruh evaluasi, bukan contoh terpilih.
domain_rows=[]
for domain,tasks in val_result.get('domains',{}).items():
    for task,result in tasks.items():
        fields=('map50','map50_95','precision','recall','images') if task=='det' else ('mean_iou','dice','f1','images')
        domain_rows.append({'domain':domain,'task':task,**{k:result.get(k) for k in fields}})
domains=pd.DataFrame(domain_rows)
domains.to_csv(Path(cfg.drive)/'evaluation/val/domain_comparisons.csv',index=False)
display(domains)


## 11. Overfitting Analysis

Ukur generalization gap menggunakan checkpoint yang sama pada **seluruh train tanpa augmentasi** dan validation. Ini komputasi tambahan yang memang mahal; tidak memakai subset tersembunyi. Train metrics bersifat diagnostik, bukan estimasi generalisasi. Validation telah memengaruhi model selection, sehingga held-out test berlabel tetap lebih kuat bila tersedia.

Kurva loss, gap metric, performa tiap kelas/domain, dan contoh false negative/false positive digunakan bersama. Gap positif saja tidak membuktikan overfitting; perubahan distribusi, kelas langka, versi anotasi, dan missing labels juga dapat menjelaskan perbedaan. Tidak ada jaminan zero overfitting.

In [ ]:
train_result=evaluate(model,samples,cfg,device,'train')
gap_table=generalization_report(train_result,val_result,cfg)
display(gap_table)
visualize_errors(model,samples,cfg,device,'val',number=6)  # Hanya galeri; evaluasi di atas memakai SEMUA data.
errors=pd.read_csv(Path(cfg.drive)/'evaluation/val/per_image_errors.csv')
print('Jumlah gambar dalam error analysis:',len(errors))
if 'det_fn' in errors:
    display(errors.sort_values(['det_fn','det_fp'],ascending=False).head(10))  # Ringkasan tampilan, bukan pembatasan evaluasi.


## 12. Full Dataset Inference

Inferensi ditujukan kepada **setiap physical image valid yang dapat diakses**, mencakup train, validation, test, gambar tanpa label, salinan duplikat, dan gambar yang dikarantina dari supervised learning. Setiap gambar mendapat box JSON, semantic PNG, lane PNG, drivable PNG, serta overlay empat tugas. Hasil pada train tidak diklaim sebagai held-out performance.

Default output persisten berada di `cfg.drive/predictions`. Menulis ratusan ribu file ke Drive bisa lambat dan memerlukan kuota puluhan GiB. Pilot di sel berikut hanya mengestimasi ukuran output; pilot **tidak menggantikan inferensi penuh**. Jika estimasi melebihi kuota, tambahkan kapasitas Drive atau gunakan folder eksperimen pada storage remote Colab yang cukup. Tidak ada pemotongan dataset otomatis.

Setiap output mempunyai ID stabil dari relative path dan source SHA256. Record completion ditulis terakhir; output lengkap dari model/config yang sama dapat digunakan saat resume. Gambar gagal dicatat dan cakupan diperiksa dengan assertion pada akhir proses. Isi `failed_images.json` harus ditangani sebelum menyatakan inferensi lengkap.

In [ ]:
import io
valid_records=physical_images[physical_images.valid].to_dict('records')
# Pilot ukuran saja: tersebar deterministik pada inventaris, termasuk berbagai split.
pilot_indices=np.linspace(0,len(valid_records)-1,min(24,len(valid_records)),dtype=int)
measured=[]
for i in pilot_indices:
    rgb,pred,masks,overlay=infer_one(model,valid_records[int(i)]['path'],cfg,device)
    byte_count=0
    for mask in masks.values():
        b=io.BytesIO();Image.fromarray(mask).save(b,format='PNG');byte_count+=len(b.getvalue())
    b=io.BytesIO();Image.fromarray(overlay).save(b,format='JPEG',quality=85);byte_count+=len(b.getvalue())
    byte_count+=len(json.dumps({k:v.tolist() for k,v in pred.items()}).encode())+16384  # metadata + filesystem overhead.
    measured.append(byte_count)
estimated_bytes=int(np.mean(measured)*len(valid_records)*1.5)
print('Estimasi konservatif output penuh GiB:',estimated_bytes/1024**3)
print('Jumlah gambar valid yang WAJIB diinferensi:',len(valid_records))
check_disk(cfg.drive,estimated_bytes/1024**3+3)
# Periksa kuota terbaru bila service Google Drive tersedia.
if 'credentials' in globals() and 'limit' in quota_report:
    quota=build('drive','v3',credentials=credentials,cache_discovery=False).about().get(fields='storageQuota').execute()['storageQuota']
    if 'limit' in quota:
        remaining=int(quota['limit'])-int(quota.get('usage',0))
        assert remaining>=estimated_bytes+3*1024**3, 'Kuota Drive belum cukup untuk semua keluaran; tambah ruang sebelum melanjutkan.'
write_json(Path(cfg.drive)/'reports/inference_storage_estimate.json',{'pilot_images':len(measured),'full_valid_images':len(valid_records),'estimated_bytes_with_margin':estimated_bytes})


In [ ]:
prediction_index,inference_report=infer_every_image(model,physical_images,cfg,device)
assert inference_report['all_valid_images_completed']
assert inference_report['successful_predictions']==int(physical_images.valid.sum())
print('Prediksi seluruh gambar valid selesai; invalid images tercatat terpisah.')
print('Indeks:',Path(cfg.drive)/'predictions/prediction_index.jsonl')


## 13. Prediction Visualization

Overlay menggabungkan warna semantic, direct/alternative drivable, marking lane kuning, dan bounding box beserta confidence. Mask PNG tetap berupa ID numerik, bukan gambar berwarna untuk training ulang. Galeri hanya mengambil beberapa output yang **sudah dihasilkan oleh inferensi penuh**; jumlah galeri tidak membatasi cakupan inferensi.

Kontrak label: deteksi memakai indeks 0–9 sesuai `DET_NAMES`; semantic 0–18; lane 0 background/1 marking; drivable 0 direct/1 alternative/2 background. Box berformat xyxy piksel asli.

In [ ]:
show_indices=np.linspace(0,len(prediction_index)-1,min(6,len(prediction_index)),dtype=int)
fig,axes=plt.subplots(2,3,figsize=(18,8))
for ax in axes.ravel():ax.axis('off')
for ax,index in zip(axes.ravel(),show_indices):
    item=prediction_index[int(index)]
    ax.imshow(Image.open(Path(item['folder'])/'overlay.jpg'))
    ax.set_title(item['relative'][-70:])
plt.tight_layout();plt.savefig(Path(cfg.drive)/'figures/full_inference_gallery.jpg',dpi=140);plt.show()
example=prediction_index[int(show_indices[0])]
print(json.dumps(example,indent=2))
print('Nama kelas deteksi:',DET_NAMES)


**Satu gambar jalan baru.** Isi `NEW_IMAGE` dengan path gambar yang ada di Colab/Drive; tidak perlu data dari laptop. Satu pemanggilan di bawah menghasilkan keempat tugas sekaligus. Sel ini opsional dan tidak mengubah training atau laporan evaluasi.

In [ ]:
NEW_IMAGE=None  # contoh: '/content/drive/MyDrive/road_image.jpg'
if NEW_IMAGE:
    rgb,boxes,masks,overlay=infer_one(model,NEW_IMAGE,cfg,device)
    plt.figure(figsize=(14,7));plt.imshow(overlay);plt.axis('off');plt.show()
    destination=Path(cfg.drive)/'external_image_prediction';destination.mkdir(exist_ok=True)
    write_json(destination/'boxes.json',{k:v.tolist() for k,v in boxes.items()})
    for task,mask in masks.items():Image.fromarray(mask).save(destination/(task+'.png'))
    Image.fromarray(overlay).save(destination/'overlay.jpg')


## 14. Model Export

Ekspor menyimpan state dict lengkap dan **TorchScript raw graph** empat tugas. Sel memeriksa kesesuaian tujuh output eager vs traced. Kontrak fixed shape/batch=1 dijelaskan dalam `export_contract.json`; normalisasi ada di graph, sedangkan decode FCOS, NMS, inverse letterbox dan visualisasi berada di Python. Ekspor ini tidak diklaim sebagai ONNX/TensorRT atau graph end-to-end NMS yang telah diuji.

Model dapat direkonstruksi tanpa mengunduh ulang pretrained weights dengan `FourTaskNet(pretrained=False)` lalu memuat state dict. Semua modul pendukung dan environment lock disimpan bersama eksperimen.

In [ ]:
export_dir=export_model(model,cfg,device)
print('Ekspor:',export_dir)
# Verifikasi reconstruction dari state dict, tanpa download pretrained baru.
saved=torch.load(export_dir/'model_state.pt',map_location='cpu',weights_only=False)
restored=FourTaskNet(pretrained=False,max_detections=cfg.max_detections)
restored.load_state_dict(saved['model']);restored.eval()
del restored,saved
print('Rekonstruksi state dict lengkap berhasil.')


## 15. Research Conclusions

Kesimpulan harus dihasilkan setelah eksekusi, dari audit dan metrik yang benar-benar terukur. Sel berikut mengompilasi bukti; tidak menulis kesimpulan performa sebelum eksperimen berjalan.

Hal yang perlu dibahas pada laporan penelitian:

1. Cakupan label nyata per tugas, data yang dikarantina, serta bias karena label parsial atau mask resmi tidak tersedia.
2. mAP, IoU, Dice, F1 serta support tiap kelas/domain; kondisi malam/cuaca sulit dan error objek kecil/garis tipis.
3. Gap train–validation pada checkpoint sama; test dilaporkan hanya jika GT tersedia.
4. Ablation yang masih diperlukan: empat single-task baselines pada protokol sama; shared backbone vs terpisah; uncertainty vs bobot tetap; resolusi; rasterized vs official masks. Notebook ini belum mengklaim hasil ablation yang belum dijalankan.
5. Jalankan beberapa seed sebagai eksperimen terpisah sebelum melaporkan mean/standard deviation. Satu seed yang reproducible belum mengukur variabilitas antar-run.

Fungsi lane adalah segmentasi marking biner. Keempat output tidak dengan sendirinya mengestimasi jarak metrik, prediksi tabrakan, atau kebijakan kontrol kendaraan. Keterbatasan penelitian harus dijelaskan sesuai apa yang benar-benar diukur.

In [ ]:
coverage_data=json.loads((Path(cfg.drive)/'reports/dataset_coverage.json').read_text())
summary={'experiment_status':'executed_in_this_runtime','dataset':cfg.kaggle_handle,'seed':cfg.seed,
         'dataset_fingerprint':fingerprint,'completed_training_epochs':len(history),
         'coverage':coverage_data,'validation_metrics':val_result,
         'test_metrics':test_result,'inference_coverage':inference_report,
         'generalization_gap':gap_table.to_dict('records'),'export':str(export_dir),
         'limitations':['Single seed; no ablation results claimed','Legacy detection annotation version',
                       'Lane is binary pixel segmentation; not lane-instance benchmark',
                       'Official-mask availability and polygon rasterization recorded in audit',
                       'Validation used for checkpoint selection','No guarantee of a target accuracy or zero overfitting']}
write_json(Path(cfg.drive)/'research_summary.json',summary)
lines=['# Ringkasan Eksperimen BDD4',f'Dataset: {cfg.kaggle_handle}',f'Seed: {cfg.seed}',f'Epoch selesai: {len(history)}',
       f'Inferensi berhasil: {inference_report["successful_predictions"]}/{inference_report["valid_images_expected"]}',
       'Angka metrik ada di research_summary.json dan evaluation/*/metrics.json; seluruhnya berasal dari eksekusi.',
       'Test GT status: '+test_result.get('status','unknown'),
       'Jangan membandingkan langsung hasil ini dengan paper yang memakai versi label atau definisi lane berbeda.']
(Path(cfg.drive)/'research_conclusions.md').write_text('\n\n'.join(lines))
print('\n'.join(lines))


### Referensi dan provenance

- [BDD100K: A Diverse Driving Dataset for Heterogeneous Multitask Learning](https://arxiv.org/abs/1805.04687).
- [Dokumentasi unduhan resmi, commit yang diaudit](https://github.com/bdd100k/bdd100k/blob/9ac17c6c7c51d2fc83065fccd707cd5b1882a293/doc/source/download.rst) dan [format label](https://github.com/bdd100k/bdd100k/blob/9ac17c6c7c51d2fc83065fccd707cd5b1882a293/doc/source/format.rst).
- [FCOS, implementasi torchvision 0.24.0](https://github.com/pytorch/vision/blob/v0.24.0/torchvision/models/detection/fcos.py).
- [YOLOP, repository penulis](https://github.com/hustvl/YOLOP).
- [Uncertainty weighting, Kendall et al.](https://arxiv.org/abs/1705.07115).
- [KaggleHub](https://github.com/Kaggle/kagglehub) dan [dataset Kaggle yang diminta](https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k).

Lisensi dataset tetap mengikuti [BDD100K](https://github.com/bdd100k/bdd100k/blob/9ac17c6c7c51d2fc83065fccd707cd5b1882a293/doc/source/license.rst). Source implementasi menggunakan PyTorch/torchvision dan konsep rasterisasi Scalabel; paket pendukung memuat catatan atribusi. Tidak ada dataset, credentials, atau pretrained weights yang disertakan di notebook.